# RAG over ECHR + RIS + Swiss — retrieval → abstain → generate

Three corpora stay **separate on disk** (`data/echr_*.json`, `data/ris_*.json`,
`data/swiss_*.json`), combined **only in memory** into one tagged chunk list for a single
shared index. Every chunk carries `jurisdiction`, `source`, `lang`, `id`, and (ECHR)
`section`, so citations never cross jurisdictions.

- **ECHR** (EN) judgments are split by their standard headings; chunking **targets the Court's
  own assessment ("THE LAW")** and skips quoted statutes ("RELEVANT ... LAW") and
  separate/dissenting opinions. Judgments without those headings fall back to whole-text
  chunking (nothing dropped). Window ~600 words.
- **RIS** (DE) = Rechtssatz principles (one clean principle per chunk) **+ OGH civil-senate
  full decisions** (word-chunked, genre `decision`); criminal-senate records are excluded —
  the *Entfremdung* homonym (= misappropriation) — with a printed count.
- **Swiss** (DE) = entscheidsuche.ch full decisions (text from the ES `content` field),
  word-chunked, genre `decision`.
- Runs **end-to-end with no LLM installed** (retrieval-only). CPU-only.
- Dev default caps docs per source; set `MAX_DOCS_PER_SOURCE = None` for the full corpus.

## 1. Configuration

In [ ]:
import os
from pathlib import Path

# The e5 weights already sit in the local HF cache. Without this, SentenceTransformer
# still pings huggingface.co on load and a dead network raises instead of falling back
# to the cache. Export HF_HUB_OFFLINE=0 when a *new* model must be downloaded.
os.environ.setdefault("HF_HUB_OFFLINE", "1")

DATA_DIR  = Path("../data")
ECHR_FILE = DATA_DIR / "echr_parental_alienation.json"
RIS_FILE  = DATA_DIR / "ris_parental_alienation.json"
SWISS_FILE = DATA_DIR / "swiss_parental_alienation.json"
SOURCES = [("echr", ECHR_FILE), ("ris", RIS_FILE), ("swiss", SWISS_FILE)]   # source-pluggable

EMB_MODEL = "intfloat/multilingual-e5-base"   # needs query:/passage: prefixes
GEN_MODEL = "llama3.2"                         # local 3B; qwen2.5:0.5b is faster but garbles citations
USE_LLM   = True                              # False = force retrieval-only

# chunking — ECHR/Swiss/RIS decisions word-windowed; RIS principles one chunk each
CHUNK_SIZE_WORDS   = 600     # larger window for full judgments
CHUNK_OVERLAP      = 80
MAX_CHUNKS_PER_DOC = 60

# RIS: Rechtssatz principles are always indexed (one clean principle per chunk).
# RIS_INCLUDE_DECISIONS additionally indexes the 'Text' full-decision records, restricted to
# OGH *civil* senates (senate code 'Ob'): the criminal senates ('Os') hit the Entfremdung
# homonym (= misappropriation, §§ 133 ff StGB) and the AUSL/Bsw records are Austrian-institute
# summaries of ECtHR judgments — both off-topic for the family-law probe, so they are dropped
# with a printed count, never silently.
RIS_PRINCIPLES_ONLY  = False
RIS_INCLUDE_DECISIONS = True
_RIS_CIVIL_ID = "OGH0002"            # stable_id marker: real OGH decisions (not AUSL/Bsw)

# Swiss (entscheidsuche.ch): German-language full decisions; text lives in the ES-extracted
# `content` field (attachment.content), NOT `full_text`.
SWISS_INCLUDE = True

# ECHR sectioning: skip quoted statutes + dissents; index the Court's reasoning (+facts)
ECHR_SKIP_SECTIONS = {"RELEVANT_LAW", "OPINION"}
ECHR_LAW_ONLY      = False   # True = index ONLY the "THE LAW" assessment section

# display snippet (sentence-aligned)
SNIPPET_TARGET = 2200
SNIPPET_MAX    = 2500

MAX_DOCS_PER_SOURCE = None    # dev cap; None = full corpus

# --- recall vs. grounding (they are NOT the same number, and both are reported) ---
# TOP_K chunks are retrieved and logged in full; only the chunks that fit GEN_CHAR_BUDGET
# reach the generator, and only some of those end up cited. Every answer therefore carries
# three counts -- retrieved / in-prompt / cited -- plus a provenance JSON with KWIC extracts.
TOP_K              = 50    # recall ceiling: chunks kept after the cut + MMR + per-case cap

# Similarity cut. MEASURED on this corpus (2026-08-19, 3 EN/DE queries x 400 candidates):
# e5 cosines are compressed into a ~0.04-wide band — top 0.893, rank-400 0.853 — so an
# ABSOLUTE threshold is not a usable instrument: 0.84 keeps all 400, 0.86 keeps 35-178
# depending on the query, and the tipping point moves with query language. The cut is
# therefore RELATIVE to the top hit; the absolute floor stays only as an off-topic backstop.
SCORE_MARGIN       = 0.025 # keep chunks with cos >= top_cos - margin
# Off-topic detection. The first attempt put an absolute floor on the TOP cosine (0.84 from
# n=6) and it FALSE-ABSTAINED on "When may contact be restricted against the child's expressed
# wishes?" -- a core question whose best decided judgment scored 0.8369, three thousandths
# under the floor, so all 344 candidates were cut and the system reported that no decided
# ruling states the principle. A single top hit cannot carry this decision.
#
# Re-measured over 7 on-topic and 6 off-topic questions (EN + DE, 2026-08-19). What separates
# them is the DEPTH of the candidate list, not its peak -- an on-topic question has a hundred
# passages that all look plausible; an off-topic one has a handful of accidents and then falls
# away. Score of the 100th candidate:
#     on-topic   0.8199 .. 0.8566   (min 0.8199)
#     off-topic  0.7579 .. 0.8000   (max 0.8000)
# Peak cosine overlaps (on-topic 0.845-0.885 vs off-topic up to 0.825); plateau COUNT is
# useless (on 7-205, off 2-400). The depth statistic has a 0.02 gap, so the threshold sits in
# the middle of it.
OFF_TOPIC_DEPTH_N   = 100   # rank whose score is read
OFF_TOPIC_DEPTH_MIN = 0.81  # below this at that rank, nothing in the corpus is on topic
SCORE_FLOOR         = None  # the old absolute floor on the top hit: measured harmful, off
MAX_CHUNKS_PER_CASE = 3    # case-level diversity: one long judgment cannot occupy the whole set
COMMUNICATED_CTX_K  = 5    # cap on the routed-out boilerplate shown as context

# generation budget. llama3.2 defaults to a 4096-token window in ollama and silently TRUNCATES
# a longer prompt -- which would drop sources the model is then told to cite. num_ctx is set
# explicitly and the char budget is derived from it (~3 chars/token for DE+EN), leaving
# headroom for the system prompt, the question and the answer.
GEN_NUM_CTX     = 8192
GEN_CHAR_BUDGET = GEN_NUM_CTX * 3 - 2500

# Comparative questions ("how do the ECHR, the OGH and Swiss courts each ...") cannot use a
# single ranked list. MEASURED 2026-08-19: an English question returned 50/50 ECHR chunks and
# 0 Austrian, 0 Swiss — although the corpus is 57% German — because e5 scores same-language
# passages higher and the relative cut is a narrow band. The model then had one jurisdiction's
# passages and a three-jurisdiction question, and relabelled ECHR ids as Swiss.
# Fix: when a question names 2+ jurisdictions, search EACH corpus separately and merge.
COMPARATIVE_ON       = True
COMPARATIVE_FETCH    = 300    # candidates scored inside each source before its own cut

# --- source routing: which CORPUS is the question about? -------------------------------
# Nothing but the embedding separates the three corpora unless a filter says otherwise, and
# e5 scores same-language passages higher. MEASURED 2026-08-27 on single-corpus questions:
#   "How does the Bundesgericht formulate the best interest of the child?" -> 47/50 ECHR
#   "Wie wird das Kindeswohl im oesterreichischen Recht formuliert?"       -> 27 RIS/23 Swiss
# i.e. a question about ONE legal system answered out of another one's case law -- the same
# failure the comparative split fixed for 2+ jurisdictions, still open for exactly one.
#
# A country name carries TWO readings, and conflating them loses a real distinction:
#   "in Austria" / "Austrian case law"          -> the AUSTRIAN corpus (RIS/OGH)
#   "ECHR cases against Austria" / "v. AUSTRIA" -> the ECHR corpus, respondent State AUT
# The first is a SOURCE choice, the second a metadata filter INSIDE one source. The
# discriminator is the respondent cue ("against X", "gegen X", "v. X") -- the phrasing the
# Court's own case names carry -- not a guess about intent.
SOURCE_ROUTING = True
# Institution names only. Country names are deliberately NOT listed here: they arrive through
# the learned alias map (scope_countries), so one lexicon governs both readings.
SOURCE_PATTERNS = {   # deliberately NOT matching bare "at"/"ch" (English stopwords), and DE
    # abbreviations are listed next to the EN ones because half the corpus is German
    "echr":  r"\bechr\b|\becthr\b|\begmr\b|stra(?:ss?|ß)b(?:ou|u)rg|european court"
             r"|\bemrk\b|convention on human rights|gerichtshof für menschenrechte",
    "ris":   r"\bogh\b|oberster gerichtshof|\bris\b|austrian supreme court",
    "swiss": r"bundesgericht|\bbger\b|kantonsgericht|obergericht|tribunal fédéral"
             r"|swiss federal court",
}
SOURCE_LABELS = {"echr": "European Court of Human Rights judgments",
                 "ris": "Austrian OGH case law (RIS, Rechtssätze + civil decisions)",
                 "swiss": "Swiss court decisions"}
# States we hold a national corpus for. The same state as an ECHR respondent is a DIFFERENT
# body of documents (45 judgments against Austria, 19 against Switzerland), and this pair is
# what keeps the two reachable separately instead of one shadowing the other.
NATIONAL_CORPUS       = {"AT": "ris", "CH": "swiss"}
ECHR_RESPONDENT_EQUIV = {"AT": "AUT", "CH": "CHE"}
RESPONDENT_CUE = r"against|versus|v\.|\bvs?\b|gegen|contre"   # "cases against Austria" -> ECHR

KWIC_WINDOW    = 200       # chars either side of a query anchor in the provenance extracts
KWIC_MAX_PER_CHUNK = 3     # extracts per chunk

# How sources are packed into the generation prompt. Full chunks (~3.5k chars) exhaust the
# window after ~6 sources; keyword-anchored extracts fit ~5x more CASES into the same budget.
# The trade is attribution: a bare window can read like the Court's holding when it is really
# an applicant submission or a quoted domestic ruling, so the top-ranked sources — the ones an
# answer is most likely to quote — stay FULL, and only the long tail is packed as extracts.
# MEASURED on this question, 50 retrieved chunks / 47 cases, same 22k-char budget
# (2026-08-19). The cost driver is the NUMBER of windows, not their width:
#   win=320 x2  hybrid  7 chunks /  7 cases    extract   9 /  9
#   win=240 x2  hybrid  9 chunks /  9 cases    extract  12 / 10
#   win=240 x1  hybrid 22 chunks / 19 cases    extract  38 / 35   <- default
#   win=180 x1  hybrid 26 chunks / 23 cases    extract  43 / 40
#   win=120 x1  hybrid 28 chunks / 25 cases    extract  50 / 47   (every retrieved case fits,
#                                                        but ~1 sentence of context each)
# Full chunks put 6 cases in the prompt, so the default is ~3x the case breadth. Going wider
# means paying in context per case, which is where misattribution comes from -- not free.
# Answer SHAPE. "enumerate" is what an unconstrained small model does by default: one
# paragraph per source, i.e. a walk through the retrieved list, which reads as a pile of
# case summaries rather than an answer. "synthesis" asks for one PROPOSITION per paragraph
# with every supporting id after it -- the same evidence and the same citations, grouped by
# what the sources say instead of by where they came from.
GEN_ANSWER_STYLE     = "synthesis"   # "synthesis" | "enumerate"
# Hard cap on sources in the prompt. OBSERVED cliff with llama3.2 (3B), same corpus:
#   19 sources -> coherent prose, 4 cases cited
#   21 sources -> coherent, per-jurisdiction sections held
#   22 sources -> stops synthesising, pastes source snippets verbatim
#   30 sources -> invents placeholder citations "[a][b][c]", zero resolvable ids
#   32 sources -> echoes the source list back ("[1] ECHR ... Not found in corpus")
# Breadth past the cliff does not add evidence, it destroys the answer -- and it does so while
# the CITATION COUNT goes UP, which is why this needs a cap rather than a warning.
GEN_MAX_SOURCES      = 18         # sources placed in the prompt, whatever the budget allows
GEN_PACKING          = "hybrid"   # "full" | "extract" | "hybrid"
GEN_FULL_HEAD        = 3          # hybrid: top-N sources kept as full chunks
GEN_EXTRACT_WINDOW   = 240        # chars either side of the anchor, then snapped to sentences
GEN_EXTRACT_WINDOWS  = 1          # windows per extract source
PROVENANCE_DIR = Path("../reports/retrieval_provenance")

# --- genre-aware retrieval + MMR diversity (ECHR ingestion-discipline fix) ---
# ECHR communicated cases (pending applications, no decided ruling) carry a clean
# "SUBJECT MATTER OF THE CASE" restatement that paraphrases the query and out-scores real
# merits judgments, then floods the top-k as near-duplicates. We keep them in the corpus
# (correct data for "is this topic litigated / how often") but ROUTE them out of
# law-questions. Genre is metadata derived from existing text — no re-scrape, no re-embed.
LAW_GENRE_EXCLUDE = {"communicated"}   # genres dropped for "what is the law" questions
FETCH_K           = 400                # candidate pool before floor + MMR + per-case cap.
                                       # Must stay well above TOP_K or MMR has no room to
                                       # diversify and the floor has nothing to select from.
MMR_LAMBDA        = 0.7                # relevance vs diversity (1.0 = pure relevance)
MMR_DUP_CEIL      = 0.95               # cruder fallback: hard-drop near-duplicate candidates
MMR_POOL_CEIL     = 12                 # x TOP_K: candidates entering the MMR pass after the
                                       # cut. MMR rescans the whole pool for every pick, and a
                                       # scoped search cuts against an in-scope top hit, so the
                                       # surviving pool can be thousands of chunks. Diversity
                                       # saturates long before that; the cap keeps a scoped
                                       # question interactive and is counted like any other cut.

# --- respondent-state scoping ("... where the respondent State is Romania") ---
# A vector search has no notion of "against Romania": the phrase is a METADATA restriction,
# not a topic, and e5 ranks a Norwegian or Polish contact-rights passage above a Romanian one
# whenever it is worded closer to the question. The scope is therefore applied as a hard
# filter over the same index (no re-embed), exactly like the genre filter, and every step of
# it is counted -- a recall cut the reader cannot see is not auditable.
SCOPE_BY_RESPONDENT = True    # False = a state named in a question stays topical only
# Name -> code is LEARNED from the corpus (see learn_country_aliases): every ECHR title names
# the respondent in English while the record carries the code, so the map grows with the data
# instead of being hand-written. Only ECHR titles do that, so the two national corpora -- and
# the German surface forms a DE question uses -- are declared here, as configuration.
COUNTRY_ALIASES_EXTRA = {
    "AT": {"AUSTRIA", "AUSTRIAN", "ÖSTERREICH", "OESTERREICH", "OGH"},
    "CH": {"SWITZERLAND", "SWISS", "SCHWEIZ", "SUISSE", "SVIZZERA"},
}
FETCH_K_SCOPED = None   # candidate pool while a scope filter is active; None = whole index.
                        # Post-filtering the ordinary 400-candidate pool would leave a handful
                        # of in-scope chunks (they are ~6% of the corpus); the filter is exact
                        # metadata, so the honest move is to search everything and then cut.

EMB_CACHE = DATA_DIR / "rag_echr_ris_emb_cache.npz"

print("inputs:", [str(p.name) for _, p in SOURCES])
print(f"chunk={CHUNK_SIZE_WORDS}w/{CHUNK_OVERLAP}o | ECHR skip={ECHR_SKIP_SECTIONS} law_only={ECHR_LAW_ONLY} | dev_cap={MAX_DOCS_PER_SOURCE}")
print(f"recall: top_k={TOP_K} cut=top-{SCORE_MARGIN} (floor {SCORE_FLOOR}) fetch_k={FETCH_K} "
      f"max_per_case={MAX_CHUNKS_PER_CASE} "
      f"| generation budget={GEN_CHAR_BUDGET} chars @ num_ctx={GEN_NUM_CTX}")
print(f"RIS decisions={RIS_INCLUDE_DECISIONS} (civil only) | Swiss={SWISS_INCLUDE}")
print(f"genre routing: exclude={LAW_GENRE_EXCLUDE} | MMR fetch_k={FETCH_K} lambda={MMR_LAMBDA}")
print(f"scope by respondent={SCOPE_BY_RESPONDENT} | source routing={SOURCE_ROUTING} "
      f"| answer style={GEN_ANSWER_STYLE}")

inputs: ['echr_parental_alienation.json', 'ris_parental_alienation.json', 'swiss_parental_alienation.json']
chunk=600w/80o | ECHR skip={'OPINION', 'RELEVANT_LAW'} law_only=False | dev_cap=None
RIS decisions=True (civil only) | Swiss=True
genre routing: exclude={'communicated'} | MMR fetch_k=30 lambda=0.7


## 2. Load + normalise (each source independently)
ECHR `date` cascade: `judgementdate` (timestamp) → **`ecli`** (`ECLI:CE:ECHR:YYYY:MMDD`,
verified to match judgementdate) → a date phrase in `full_text` → "" — all rendered
**YYYY-MM-DD**. `conclusion` is metadata only (Article-8 finding, never the custody
outcome). RIS keys are lowercase; principle = text between the `Rechtssatz` and
`Entscheidungstexte` labels; applied decisions kept as `applied_decisions` metadata.

In [ ]:
import json
import re
from datetime import datetime

_EN_MONTHS = {m: i for i, m in enumerate(
    ["january", "february", "march", "april", "may", "june", "july", "august",
     "september", "october", "november", "december"], 1)}
_FT_DATE = re.compile(
    r"(?:communicated on|published on|strasbourg,?|judgment\s+strasbourg|decision\s+strasbourg)"
    r"\s+(\d{1,2})\s+([A-Za-z]+)\s+(\d{4})", re.IGNORECASE)


def load_json_records(path):
    if not path.exists():
        return None
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, dict):     # RIS single-result-as-dict guard
        data = [data]
    return data


def _echr_date(r):
    # 1) judgementdate timestamp e.g. "19/01/2016 00:00:00"
    raw = (r.get("judgementdate") or "").strip()
    if raw:
        try:
            return datetime.strptime(raw.split()[0], "%d/%m/%Y").date().isoformat()
        except (ValueError, IndexError):
            pass
    # 2) ECLI encodes the date: ECLI:CE:ECHR:YYYY:MMDD...
    m = re.match(r"ECLI:CE:ECHR:(\d{4}):(\d{2})(\d{2})", r.get("ecli", "") or "")
    if m:
        y, mo, d = m.groups()
        if 1 <= int(mo) <= 12 and 1 <= int(d) <= 31:
            return f"{y}-{mo}-{d}"
    # 3) a date phrase in the body ("Communicated on 24 August 2015", "STRASBOURG 19 January 2016")
    fm = _FT_DATE.search((r.get("full_text", "") or "")[:4000])
    if fm:
        d, mon, y = fm.groups()
        mi = _EN_MONTHS.get(mon.lower())
        if mi:
            return f"{y}-{mi:02d}-{int(d):02d}"
    return ""


def split_ris_principle(full_text):
    lines = (full_text or "").split("\n")

    def find(label):
        for i, l in enumerate(lines):
            if l.strip() == label:
                return i
        return -1

    i_rs = find("Rechtssatz")
    if i_rs == -1:
        return "", ""
    i_et = find("Entscheidungstexte")
    i_ecli = find("European Case Law Identifier")
    end = i_et if i_et != -1 else (i_ecli if i_ecli != -1 else len(lines))
    principle = "\n".join(lines[i_rs + 1:end]).strip()
    applied = ""
    if i_et != -1:
        a_end = i_ecli if i_ecli != -1 else len(lines)
        applied = "\n".join(lines[i_et + 1:a_end]).strip()
    return principle, applied


def normalise_echr(r):
    iid = r.get("itemid", "") or ""
    return {
        "id": iid,
        "title": r.get("docname", "") or iid,
        "text": r.get("full_text", "") or "",
        "jurisdiction": "ECHR",
        "source": "echr",
        "country": r.get("respondent", "") or "",
        "date": _echr_date(r),
        "lang": "en",
        "url": f"https://hudoc.echr.coe.int/eng?i={iid}" if iid else "",
        "matched_keywords": r.get("matched_keywords") or [],
        "applied_decisions": [],
        "genre_hint": None,                                    # ECHR genre from doctype/markers
        "meta": {"conclusion": r.get("conclusion", ""), "article": r.get("article", ""),
                 "importance": r.get("importance", ""), "appno": r.get("appno", ""),
                 "doctype": r.get("doctype", ""),            # HEJUD/HEDEC/HECOM -> genre
                 "doctypebranch": r.get("doctypebranch", "")},
    }


def normalise_ris(r):
    """RIS Rechtssatz -> genre 'principle' (the distilled principle text only);
    RIS 'Text' full decision -> genre 'decision' (whole decision text, word-chunked)."""
    rid = r.get("id", "") or ""
    gericht = r.get("gericht", "") or "OGH"
    rsnum = (r.get("rechtssatznummern") or "").strip()
    gz = (r.get("geschaeftszahl") or "").split(";")[0].strip()
    is_principle = r.get("dokumenttyp") == "Rechtssatz"
    if is_principle:
        text, _ = split_ris_principle(r.get("full_text", ""))
        title = (f"{gericht} {rsnum}" if rsnum else f"{gericht} {gz}").strip()
    else:
        text = r.get("full_text", "") or ""
        title = f"{gericht} {gz}".strip()
    return {
        "id": rid,
        "title": title or rid,
        "text": text,
        "jurisdiction": "AT (OGH)",
        "source": "ris",
        "country": "AT",
        "date": (r.get("entscheidungsdatum") or "")[:10],
        "lang": "de",
        "url": r.get("content_url_html", "") or r.get("source_url", ""),
        "matched_keywords": r.get("matched_keywords") or [],
        "applied_decisions": r.get("entscheidungstexte") or [],
        "genre_hint": "principle" if is_principle else "decision",
        "meta": {"geschaeftszahl": r.get("geschaeftszahl", ""), "rechtssatznummern": rsnum,
                 "rechtsgebiete": r.get("rechtsgebiete", ""),
                 "dokumenttyp": r.get("dokumenttyp", "")},
    }


def normalise_swiss(r):
    """entscheidsuche.ch record: full decision text is in `content` (ES attachment.content)."""
    rid = r.get("stable_id", "") or r.get("Signatur", "") or ""
    return {
        "id": rid,
        "title": (r.get("title") or rid).strip(),
        "text": r.get("content", "") or "",
        "jurisdiction": "CH",
        "source": "swiss",
        "country": "CH",
        "date": (r.get("Datum") or "")[:10],
        "lang": r.get("lang", "de") or "de",
        "url": r.get("source_url", "") or r.get("content_url", ""),
        "matched_keywords": r.get("matched_keywords") or [],
        "applied_decisions": [],
        "genre_hint": "decision",
        "meta": {"canton": r.get("canton", ""), "court_type": r.get("court_type", ""),
                 "hierarchy": r.get("hierarchy", ""), "reference": r.get("reference", ""),
                 "collection": r.get("collection", "")},
    }


NORMALISERS = {"echr": normalise_echr, "ris": normalise_ris, "swiss": normalise_swiss}
print("normalisers ready:", list(NORMALISERS))

normalisers ready: ['echr', 'ris', 'swiss']


## 3. ECHR section splitter + sentence snippet + chunking
ECHR headings are uppercase and glued to the text (`THE LAWI. ALLEGED VIOLATION…`), so they
are matched by regex. A judgment with a `THE LAW` heading is sectioned; otherwise it falls
back to whole-text. Display snippets are cut on a sentence boundary (~2000–2500 chars, never
mid-word); the full chunk text is preserved for the LLM context.

In [ ]:
# uppercase, glued, case-sensitive anchors
ECHR_ANCHORS = [
    (re.compile(r"PROCEDURE(?=[0-9IVX])"), "PROCEDURE"),
    (re.compile(r"THE FACTS(?=[0-9IVX]|\s)"), "FACTS"),
    (re.compile(r"THE CIRCUMSTANCES OF THE CASE"), "FACTS"),
    (re.compile(r"RELEVANT (?:DOMESTIC|LEGAL|INTERNATIONAL|EUROPEAN|COMPARATIVE|COUNCIL)"
                r"[A-Z ]{0,40}?(?:LAW|FRAMEWORK|MATERIAL|PRACTICE|TEXT)"), "RELEVANT_LAW"),
    (re.compile(r"(?:AS TO )?THE LAW(?![a-z])|THE COURT[^A-Za-z]{0,2}S ASSESSMENT"), "LAW"),
    (re.compile(r"FOR THESE REASONS"), "OPERATIVE"),
    (re.compile(r"(?:JOINT |PARTLY )?(?:DISSENTING|SEPARATE|CONCURRING) OPINION"), "OPINION"),
]


def echr_sections(full_text):
    # returns list of (label, text) or None if no 'THE LAW' heading is found
    marks = sorted((m.start(), lab) for rx, lab in ECHR_ANCHORS for m in rx.finditer(full_text))
    if not marks or not any(lab == "LAW" for _, lab in marks):
        return None
    segs = []
    if marks[0][0] > 0:
        segs.append(("HEADER", full_text[:marks[0][0]]))
    for i, (pos, lab) in enumerate(marks):
        end = marks[i + 1][0] if i + 1 < len(marks) else len(full_text)
        segs.append((lab, full_text[pos:end]))
    return segs


def _echr_keep(section):
    if section == "unparsed":
        return True                      # fallback: keep everything
    if ECHR_LAW_ONLY:
        return section == "LAW"
    return section not in ECHR_SKIP_SECTIONS


_WS = re.compile(r"\S+")
_SENT = re.compile(r"(?<=[.!?])\s+")


def chunk_words(text, size=CHUNK_SIZE_WORDS, overlap=CHUNK_OVERLAP, max_chunks=MAX_CHUNKS_PER_DOC):
    words = _WS.findall(text or "")
    if not words:
        return []
    if len(words) <= size:
        return [" ".join(words)]
    step = max(1, size - overlap)
    out = []
    for start in range(0, len(words), step):
        out.append(" ".join(words[start:start + size]))
        if start + size >= len(words):
            break
    return out[:max_chunks] if max_chunks else out


def sentence_snippet(text, target=SNIPPET_TARGET, hard_max=SNIPPET_MAX):
    text = (text or "").strip()
    if len(text) <= target:
        return text
    out = ""
    for s in _SENT.split(text):
        if not out:
            out = s
        elif len(out) + 1 + len(s) <= hard_max:
            out = out + " " + s
        else:
            break
        if len(out) >= target:
            break
    if len(out) > hard_max:                 # single huge sentence -> cut on a word boundary
        cut = out[:hard_max]
        out = cut[:cut.rfind(" ")] if " " in cut else cut
    return out + ("…" if len(out) < len(text) else "")


def build_chunks(records):
    """ECHR -> section-aware pieces; RIS principle -> one clean chunk;
    RIS decision / Swiss decision -> word-windowed whole text (no ECHR-style sections)."""
    chunks = []
    for rec in records:
        genre = assign_genre(rec)                              # first-class chunk field
        if rec["source"] == "echr":
            secs = echr_sections(rec["text"])
            if secs is None:
                pieces = [("unparsed", rec["text"])]            # safe fallback
            else:
                pieces = [(lab, seg) for lab, seg in secs if _echr_keep(lab)]
        elif genre == "principle":
            pieces = [("principle", rec["text"])]
        else:
            pieces = [("decision", rec["text"])]
        idx = 0
        for section, segtext in pieces:
            subs = [segtext] if section == "principle" else chunk_words(segtext)
            for piece in subs:
                if not piece.strip():
                    continue
                chunks.append({
                    "chunk_id": f"{rec['source']}:{rec['id']}:{idx}",
                    "doc_id": rec["id"], "source": rec["source"],
                    "jurisdiction": rec["jurisdiction"], "section": section,
                    "genre": genre,
                    "title": rec["title"], "url": rec["url"], "date": rec["date"],
                    "lang": rec["lang"], "country": rec["country"],
                    "matched_keywords": rec["matched_keywords"],
                    "applied_decisions": rec.get("applied_decisions", []),
                    "text": piece,
                })
                idx += 1
    return chunks


print("section splitter + chunker ready")

section splitter + chunker ready


## 3b. Genre — a first-class chunk field (reusable helper)
ECHR documents come in three **genres** that differ in answer-value for a *"what is the law"*
question:

- **`merits`** — decided judgments (`doctype=HEJUD`). The Court states the principle. **Wanted.**
- **`admissibility`** — admissibility decisions (`HEDEC`). Often principle-bearing. **Kept.**
- **`communicated`** — *pending* applications (`HECOM`): a clean `SUBJECT MATTER OF THE CASE`
  restatement + `QUESTIONS TO THE PARTIES`, **no decided ruling**. High query-similarity,
  zero answer-value, near-duplicate templates that flood the top-k. **Routed out of law-questions
  (not deleted)** — they are the *correct* data for "is this topic being litigated / how often".

Genre is derived from existing fields/text (no re-scrape, no re-embed): primary signal is the
HUDOC `doctype`, with a textual-marker fallback (`Communicated on`, `SUBJECT MATTER OF THE
CASE`, `QUESTIONS TO THE PARTIES`) for records lacking the field. RIS carries **no** ECHR
genre — its Rechtssatz principles are tagged `principle` (this fix is ECHR-only). The next
cell cross-checks the marker detector against the existing *unparsed* parser-flag before the
genre is wired into retrieval.

In [ ]:
# --- reusable ECHR genre helper (dependency-free: needs only `re`) ---
# Copy-paste-able into the diachronic / framing notebooks; classifies an ECHR document into
# {merits, admissibility, communicated} from its HUDOC doctype and/or its body text.

ECHR_GENRES = ("merits", "admissibility", "communicated", "other")
LOW_INFO_GENRES = {"communicated"}        # genres that carry no decided principle

_DOCTYPE_GENRE = {"HEJUD": "merits", "HEDEC": "admissibility", "HECOM": "communicated"}
_GENRE_LAW_RE = re.compile(r"(?:AS TO )?THE LAW(?![a-z])|THE COURT[^A-Za-z]{0,2}S ASSESSMENT")
_RE_COMMUNICATED_ON = re.compile(r"Communicated on")
_MARK_SUBJECT = "SUBJECT MATTER OF THE CASE"
_MARK_QUESTIONS = "QUESTIONS TO THE PARTIES"


def echr_genre_from_markers(text):
    """Genre from body-text markers only (no doctype) — used for the cross-check."""
    t = text or ""
    if _RE_COMMUNICATED_ON.search(t) or _MARK_QUESTIONS in t:
        return "communicated"
    if _MARK_SUBJECT in t and not _GENRE_LAW_RE.search(t):
        return "communicated"            # subject-matter restatement with no 'THE LAW' = pending
    if _GENRE_LAW_RE.search(t):
        return "merits"                  # has the Court's assessment; HEDEC refined via doctype
    return "other"


def echr_genre(text, doctype=""):
    """Primary classifier: trust the HUDOC doctype, fall back to body markers."""
    g = _DOCTYPE_GENRE.get((doctype or "").upper())
    return g if g else echr_genre_from_markers(text)


def assign_genre(rec):
    """Genre for any corpus record. ECHR -> merits/admissibility/communicated;
    non-ECHR sources carry a genre_hint from their normaliser:
    'principle' (RIS Rechtssatz) or 'decision' (RIS Text / Swiss full decision)."""
    if rec.get("source") != "echr":
        return rec.get("genre_hint") or "principle"
    return echr_genre(rec.get("text", ""), rec.get("meta", {}).get("doctype", ""))


print("genre helper ready:", ECHR_GENRES, "| low-info:", LOW_INFO_GENRES)

genre helper ready: ('merits', 'admissibility', 'communicated', 'other') | low-info: {'communicated'}


## 4. Build the in-memory corpus + report (dates, sections)

In [ ]:
from collections import Counter

records, chunks = [], []
DATA_PRESENT = all(p.exists() for _, p in SOURCES)

# OGH criminal senate (the Entfremdung = misappropriation homonym). A \b after "Os"
# does not fire on API-format Geschäftszahlen ("13Os7/06p"), only on the whitespace-
# padded scraped form ("13  Os    7/06p") — match the following digit instead.
_RIS_CRIMINAL = re.compile(r"\d{1,3}\s*Os\s*\d", re.IGNORECASE)

if DATA_PRESENT:
    for src, path in SOURCES:
        raw = load_json_records(path) or []
        if src == "ris":
            n_all = len(raw)
            principles = [r for r in raw if r.get("dokumenttyp") == "Rechtssatz"]
            if RIS_PRINCIPLES_ONLY or not RIS_INCLUDE_DECISIONS:
                raw = principles
                print(f"  ris : {len(raw)} principles (dropped {n_all - len(raw)} 'Text' decisions)")
            else:
                texts = [r for r in raw if r.get("dokumenttyp") == "Text"]
                civil = [r for r in texts
                         if _RIS_CIVIL_ID in (r.get("id") or "")
                         and not _RIS_CRIMINAL.search((r.get("geschaeftszahl") or "").split(";")[0])]
                print(f"  ris : {len(principles)} principles + {len(civil)} civil decisions "
                      f"(dropped {len(texts) - len(civil)} criminal-senate/AUSL 'Text' records "
                      f"— Entfremdung homonym / ECtHR summaries)")
                raw = principles + civil
        if src == "swiss":
            if not SWISS_INCLUDE:
                print("  swiss: skipped (SWISS_INCLUDE=False)")
                continue
            n_all = len(raw)
            raw = [r for r in raw
                   if "rechenschaftsbericht" not in str(r.get("title", "")).lower()]
            if n_all - len(raw):
                print(f"  swiss: dropped {n_all - len(raw)} Rechenschaftsbericht records "
                      f"(court annual reports, not case law — multi-case digests that "
                      f"flood the top-k)")
        if MAX_DOCS_PER_SOURCE:
            raw = raw[:MAX_DOCS_PER_SOURCE]
        recs = [NORMALISERS[src](r) for r in raw]
        n_empty = sum(1 for r in recs if not (r["text"] or "").strip())
        if n_empty:
            print(f"  !! {src}: {n_empty} records with empty text (kept out of the index)")
            recs = [r for r in recs if (r["text"] or "").strip()]
        records += recs
        print(f"  {src:5s}: {len(recs)} records from {path.name}")

    # reports the task asks for
    echr_recs = [r for r in records if r["source"] == "echr"]
    if echr_recs:
        dated = sum(1 for r in echr_recs if r["date"])
        parsed = sum(1 for r in echr_recs if echr_sections(r["text"]) is not None)
        print(f"\nECHR dates: {dated}/{len(echr_recs)} have YYYY-MM-DD, {len(echr_recs) - dated} n.d.")
        print(f"ECHR sections: {parsed}/{len(echr_recs)} parsed; {len(echr_recs) - parsed} fell back to whole-text")

    chunks = build_chunks(records)
    print(f"\nrecords: {len(records)}  ->  chunks: {len(chunks)}")
    print("chunks by jurisdiction:", dict(Counter(c["jurisdiction"] for c in chunks)))
    print("chunks by language    :", dict(Counter(c["lang"] for c in chunks)))
    print("ECHR chunks by section :", dict(Counter(c["section"] for c in chunks if c["source"] == "echr")))
else:
    for _, p in SOURCES:
        if not p.exists():
            print("missing:", p)

  echr : 1116 records from echr_parental_alienation.json
  ris : 38 principles + 479 civil decisions (dropped 31 criminal-senate/AUSL 'Text' records — Entfremdung homonym / ECtHR summaries)
  ris  : 517 records from ris_parental_alienation.json
  swiss: dropped 24 Rechenschaftsbericht records (court annual reports, not case law — multi-case digests that flood the top-k)
  swiss: 2007 records from swiss_parental_alienation.json

ECHR dates: 1114/1116 have YYYY-MM-DD, 2 n.d.
ECHR sections: 873/1116 parsed; 243 fell back to whole-text

records: 3640  ->  chunks: 42055
chunks by jurisdiction: {'ECHR': 15136, 'AT (OGH)': 1931, 'CH': 24988}
chunks by language    : {'en': 15136, 'de': 26919}
ECHR chunks by section : {'HEADER': 1508, 'LAW': 6952, 'PROCEDURE': 476, 'FACTS': 4567, 'OPERATIVE': 580, 'unparsed': 1053}


### 4b. Inspect genre — distribution + cross-check the parser-flag (before wiring it in)
Prints genre per source, then **cross-tabulates** the existing *unparsed* parser-flag
(`echr_sections() is None`) against the marker-based `communicated` detector, and reports any
disagreement. If the flag and the markers diverge badly the genre is unreliable and the run
stops here for inspection rather than silently filtering.

In [ ]:
GENRE_RELIABLE = True   # set False by the cross-check below if flag vs markers disagree badly

if records:
    # genre per source — at the document level and the chunk level
    print("GENRE per source (documents):")
    for src in ("echr", "ris", "swiss"):
        srecs = [r for r in records if r["source"] == src]
        if srecs:
            print(f"  {src:5s}: {dict(Counter(assign_genre(r) for r in srecs))}")
    print("GENRE per source (chunks):")
    for src in ("echr", "ris", "swiss"):
        d = dict(Counter(c['genre'] for c in chunks if c['source'] == src))
        if d:
            print(f"  {src:5s}: {d}")

    echr_recs = [r for r in records if r["source"] == "echr"]

    # --- cross-tab: existing unparsed parser-flag  ×  marker-based communicated detector ---
    print("\nCross-check (ECHR docs): unparsed-flag  ×  marker detector")
    ct = Counter()
    disagree_unparsed_not_comm, disagree_comm_parsed = [], []
    for r in echr_recs:
        unparsed = echr_sections(r["text"]) is None              # the existing parser flag
        comm_marker = echr_genre_from_markers(r["text"]) == "communicated"
        ct[(unparsed, comm_marker)] += 1
        if unparsed and not comm_marker:
            disagree_unparsed_not_comm.append(r)                 # unparsed but a real judgment
        if comm_marker and not unparsed:
            disagree_comm_parsed.append(r)                       # 'communicated' yet parsed
    print(f"  unparsed=True  & communicated-marker=True : {ct[(True, True)]:4d}   (agree: pending)")
    print(f"  unparsed=False & communicated-marker=False: {ct[(False, False)]:4d}   (agree: decided)")
    print(f"  unparsed=True  & communicated-marker=False: {ct[(True, False)]:4d}   <- judgments the parser missed")
    print(f"  unparsed=False & communicated-marker=True : {ct[(False, True)]:4d}   <- 'communicated' yet parsed")

    n_dis = len(disagree_unparsed_not_comm) + len(disagree_comm_parsed)
    n_comm = sum(1 for r in echr_recs if echr_genre_from_markers(r["text"]) == "communicated")
    print(f"\n  communicated (markers): {n_comm} | disagreements with parser-flag: {n_dis}")
    for r in disagree_unparsed_not_comm[:5]:
        print(f"    unparsed-but-not-communicated: {r['title'][:55]} (doctype={r['meta'].get('doctype')})")

    # verdict: the parser-flag is a PROXY; genre must come from doctype+markers, not the flag.
    # Unreliable only if marker-communicated and the explicit doctype=HECOM diverge.
    hecom = sum(1 for r in echr_recs if (r["meta"].get("doctype") or "").upper() == "HECOM")
    n_doctype_vs_marker = sum(
        1 for r in echr_recs
        if ((r["meta"].get("doctype") or "").upper() == "HECOM")
        != (echr_genre_from_markers(r["text"]) == "communicated"))
    print(f"\n  doctype=HECOM: {hecom} | doctype vs marker disagreements: {n_doctype_vs_marker}")
    GENRE_RELIABLE = n_doctype_vs_marker <= max(3, int(0.02 * len(echr_recs)))
    if GENRE_RELIABLE:
        print("  VERDICT: genre is reliable (doctype agrees with markers). "
              "Using doctype as primary; the unparsed parser-flag is a proxy only "
              f"({len(disagree_unparsed_not_comm)} real judgments are unparsed and must NOT be dropped).")
    else:
        print("  !! VERDICT: doctype and markers disagree beyond tolerance — STOP and inspect "
              "before wiring genre into retrieval. (Do not filter on an unreliable signal.)")

    # non-ECHR sanity: German sources must never inherit the ECHR communicated genre
    for src in ("ris", "swiss"):
        srecs = [r for r in records if r["source"] == src]
        if not srecs:
            continue
        s_comm = sum(1 for r in srecs if echr_genre_from_markers(r["text"]) == "communicated")
        print(f"\n{src.upper()} sanity: {len(srecs)} records, genres "
              f"{dict(Counter(assign_genre(r) for r in srecs))}; "
              f"ECHR communicated-markers present = {s_comm} (expected 0 — fix is ECHR-only).")
else:
    print("No records loaded — genre report skipped.")

GENRE per source (documents):
  echr : {'admissibility': 311, 'merits': 567, 'communicated': 238}
  ris  : {'principle': 38, 'decision': 479}
  swiss: {'decision': 2007}
GENRE per source (chunks):
  echr : {'admissibility': 2769, 'merits': 11418, 'communicated': 949}
  ris  : {'principle': 38, 'decision': 1893}
  swiss: {'decision': 24988}

Cross-check (ECHR docs): unparsed-flag  ×  marker detector
  unparsed=True  & communicated-marker=True :  225   (agree: pending)
  unparsed=False & communicated-marker=False:  873   (agree: decided)
  unparsed=True  & communicated-marker=False:   18   <- judgments the parser missed
  unparsed=False & communicated-marker=True :    0   <- 'communicated' yet parsed

  communicated (markers): 225 | disagreements with parser-flag: 18
    unparsed-but-not-communicated: I.S. v. GERMANY (doctype=HECOM)
    unparsed-but-not-communicated: CARSTOIU v. ROMANIA (doctype=HECOM)
    unparsed-but-not-communicated: LOPEZ GUIO v. SLOVAKIA (doctype=HECOM)
    unparsed

## 5. Embed + FAISS index (multilingual-e5-base, CPU, exact cosine)
Cached to disk; numpy exact-cosine fallback if `faiss` isn't installed.

In [ ]:
import numpy as np

_embedder = None


def _get_embedder():
    global _embedder
    if _embedder is None:
        from sentence_transformers import SentenceTransformer
        _embedder = SentenceTransformer(EMB_MODEL, device="cpu")
    return _embedder


def embed_passages(texts, progress=True):
    m = _get_embedder()
    return m.encode([f"passage: {t}" for t in texts], batch_size=16, convert_to_numpy=True,
                    normalize_embeddings=True, show_progress_bar=progress).astype("float32")


def embed_query(q):
    m = _get_embedder()
    return m.encode([f"query: {q}"], convert_to_numpy=True,
                    normalize_embeddings=True).astype("float32")


class _NumpyExactIndex:
    # exact cosine via brute-force dot product (identical to faiss.IndexFlatIP)
    def __init__(self, mat):
        self._m = mat
        self.ntotal = int(mat.shape[0])

    def search(self, q, k):
        sims = (q @ self._m.T)[0]
        k = min(k, self._m.shape[0])
        top = np.argpartition(-sims, k - 1)[:k]
        top = top[np.argsort(-sims[top])]
        return sims[top][None, :], top[None, :]


EMBED_CHECKPOINT_EVERY = 2000    # chunks per cache save (a CPU run of hours must survive a kill)


def _load_cached_rows():
    """Incremental cache: {chunk_id: row}. Migrates the legacy whole-corpus cache
    (keys 'key'+'emb', ECHR + RIS-principle chunks in build order) on first load."""
    if not EMB_CACHE.exists():
        return {}
    cached = np.load(EMB_CACHE, allow_pickle=True)
    if "ids" in cached.files:                     # current per-chunk format
        ids = [str(x) for x in cached["ids"]]
        return dict(zip(ids, cached["emb"].astype("float32")))
    # legacy format: reconstruct the old id order = ECHR chunks + RIS principle chunks,
    # in current corpus order (ECHR chunking and RIS principle ids are unchanged).
    legacy_ids = [c["chunk_id"] for c in chunks
                  if c["source"] == "echr" or (c["source"] == "ris" and c["genre"] == "principle")]
    emb = cached["emb"].astype("float32")
    if len(legacy_ids) == emb.shape[0]:
        print(f"migrated legacy cache: {emb.shape[0]} vectors reused")
        return dict(zip(legacy_ids, emb))
    print(f"legacy cache mismatch ({emb.shape[0]} vectors vs {len(legacy_ids)} expected) — ignoring it")
    return {}


def _save_cache(cache):
    ids = list(cache.keys())
    np.savez(EMB_CACHE, ids=np.array(ids), emb=np.stack([cache[i] for i in ids]))


def build_index(chunk_list):
    if not chunk_list:
        return None
    cache = _load_cached_rows()
    ids = [c["chunk_id"] for c in chunk_list]
    missing = [i for i, cid in enumerate(ids) if cid not in cache]
    print(f"cache: {len(ids) - len(missing)}/{len(ids)} chunks already embedded | to embed: {len(missing)}")
    if missing:
        for b0 in range(0, len(missing), EMBED_CHECKPOINT_EVERY):
            batch = missing[b0:b0 + EMBED_CHECKPOINT_EVERY]
            new_emb = embed_passages([chunk_list[i]["text"] for i in batch], progress=False)
            for i, row in zip(batch, new_emb):
                cache[ids[i]] = row
            _save_cache(cache)
            print(f"  embedded {min(b0 + len(batch), len(missing))}/{len(missing)} new chunks "
                  f"(checkpointed -> {EMB_CACHE.name})")
    emb = np.stack([cache[cid] for cid in ids]).astype("float32")
    try:
        import faiss
        idx = faiss.IndexFlatIP(emb.shape[1])
        idx.add(emb)
        print("index backend: faiss IndexFlatIP (exact)")
        return idx
    except ImportError:
        print("faiss not installed -> numpy exact-cosine fallback (identical results)")
        return _NumpyExactIndex(emb)


def index_matrix(idx):
    # The same normalised e5 vectors already in the flat index, kept in memory for MMR
    # (no re-embed). faiss -> reconstruct_n; numpy fallback -> the stored matrix.
    if idx is None:
        return None
    if hasattr(idx, "_m"):                       # numpy fallback
        return idx._m
    return idx.reconstruct_n(0, idx.ntotal)      # faiss IndexFlatIP


index = None
EMB_MATRIX = None
if chunks:
    try:
        index = build_index(chunks)
        EMB_MATRIX = index_matrix(index)
        print(f"index ready: {index.ntotal} vectors | MMR matrix: {EMB_MATRIX.shape}")
    except ImportError as e:
        print(f"embedding library missing ({e}) -> pip install sentence-transformers")
else:
    print("no chunks — skipping index")

cache: 30295/42055 chunks already embedded | to embed: 11760
/Users/maksimsmirnov/Desktop/thesis/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
  embedded 2000/11760 new chunks (checkpointed -> rag_echr_ris_emb_cache.npz)
  embedded 4000/11760 new chunks (checkpointed -> rag_echr_ris_emb_cache.npz)
  embedded 6000/11760 new chunks (checkpointed -> rag_echr_ris_emb_cache.npz)
  embedded 8000/11760 new chunks (checkpointed -> rag_echr_ris_emb_cache.npz)
  embedded 10000/11760 new chunks (checkpointed -> rag_echr_ris_emb_cache.npz)
  embedded 11760/11760 new chunks (checkpointed -> rag_echr_ris_emb_cache.npz)
faiss not installed -> numpy exact-cosine fallback (identical results)
index ready: 42055 vectors | MMR matrix: (42055, 768)


## 6. Retrieval — source routing, genre-aware filtering, MMR diversity
Four stages, all on the **existing** index/vectors (no re-embed, no rebuild):
1. **Fetch wide** — search `fetch_k` (~30) candidates from the flat index.
2. **Genre filter** — drop `genre_filter` genres (default `communicated`) from the candidate
   set *after* the search. Override to `None`/`set()` to include them (e.g. litigation-volume
   questions).
3. **Source route** — if the question names a legal system ("in Austria", "the Bundesgericht",
   "the ECHR"), keep only that corpus. Nothing but the embedding separates the three corpora
   otherwise, and e5 scores same-language passages higher, so an English question about a
   Swiss court came back 47/50 ECHR (measured 2026-08-27) — a question about one legal system
   answered out of another one's case law. The route is decided by `route_sources()`: corpus
   names come from `SOURCE_PATTERNS`, country names from the same learned alias map the
   respondent scope uses, and a country routes to **its own corpus** unless the question uses
   a respondent cue. Two or more systems named is a comparison and goes to
   `retrieve_per_source()` instead of a single filter.
   ```
   "in Austria" / "österreichischen Recht"   -> RIS/OGH corpus, no respondent filter
   "ECHR cases against Austria" / "v. AUT"   -> ECHR corpus, respondent scope AUT
   "against Romania"                          -> ECHR corpus (no national corpus held), ROU
   ```
4. **Respondent scope** — if the question names a State ("where the respondent is Romania"),
   drop every candidate whose respondent code does not match. Applied *before* the score cut,
   because that cut is relative to the top hit: computed on the unfiltered pool, a Norwegian
   passage sets the bar and the Romanian ones fall below it. The name→code map is **learned
   from ECHR titles**, not hardcoded (`learn_country_aliases()`), and a two-State case
   (`BGR;ROU`) is in scope for both.
5. **MMR re-rank** — greedily pick `k` (~6): each next pick maximises
   `λ·sim(query) − (1−λ)·max sim(already-picked)`, reusing the in-memory e5 vectors
   (`EMB_MATRIX`). Kills the near-duplicate boilerplate clusters. A cruder hard-dedup
   (`cos > MMR_DUP_CEIL`) is available as a fallback.

With a source or scope filter on, the candidate pool is widened to the whole index (`FETCH_K_SCOPED`):
the filter is exact metadata, so recall inside the scope should not depend on how many
out-of-scope chunks happened to outrank it in the top-`fetch_k`.

The **absolute floor moves in front of both filters**, and only the relative margin runs
behind it. The floor was calibrated on corpus-wide top cosines, but a single respondent State
draws its maximum from far fewer documents and therefore scores lower on the *same* question
(measured: top 0.885 corpus-wide vs 0.837 inside `ROU` for "best interests of the child").
Left behind the filter it deletes every in-scope passage and reports an on-topic question as
off-topic. Asking it before the filter keeps its actual meaning — *is this question about this
corpus at all* — which is a judgment the scope should not change. Unscoped questions are
unaffected: with no filter the two orders are identical.

Each hit keeps `score, id, jurisdiction, country, date, section, genre, url, title, snippet`
and the full `text` (LLM context).

In [ ]:
# --- respondent-state scope: name -> code, LEARNED from the corpus ---------------------
# Hardcoding 46 Council-of-Europe states would be a lexicon to maintain (and would go stale
# the moment the corpus is extended). Every ECHR title already names the respondent in
# English -- "CASE OF SIMONA MIHAELA DOBRE v. ROMANIA" -- while the record carries the code
# ("ROU"), so the mapping is read off that pairing. The two national corpora carry no such
# title, so their names (and the German forms a DE question uses) come from
# COUNTRY_ALIASES_EXTRA.
_COUNTRY_ALIASES = None      # NAME -> CODE
_COUNTRY_NAME_RE = None      # alternation over the names, longest first
_RESPONDENT_CUE_RE = None    # the same names, but only where "against"/"v."/"gegen" precedes
_COUNTRY_CODES = None
# noun + the adjectival endings EN and DE build on it: AUSTRIA/AUSTRIAN, OESTERREICH/
# OESTERREICHISCHEN, SCHWEIZ/SCHWEIZERISCHE. Bounded (\w{0,3}) so it cannot run into the
# next word.
_NAME_SUFFIX = r"(?:N|(?:ER)?ISCH\w{0,3})?"

_V_SPLIT = re.compile(r"\bv(?:s|\.)?\s+", re.IGNORECASE)      # "X v. ROMANIA", "X vs ROMANIA"
_TITLE_PAREN = re.compile(r"\([^)]*\)|\[[^\]]*\]")            # "(No. 2)", "(dec.)"
_NAME_OK = re.compile(r"[A-Z\u00c0-\u00de][A-Z\u00c0-\u00de' \-]{2,39}$")
# codes that are also ordinary words -- matching them in a question would scope a sentence
# containing "and" to Andorra. Names still reach those states.
_CODE_STOPWORDS = {"AND", "AT", "CH", "IN", "IT", "IS", "NO", "OR", "DE", "AM", "SO", "AN"}


def _names_in(part):
    out = []
    for n in re.split(r"\band\b|&|,", part, flags=re.IGNORECASE):
        n = re.sub(r"^THE\s+", "", n.strip(" .;-").upper())   # "THE NETHERLANDS" == "NETHERLANDS"
        if _NAME_OK.match(n):
            out.append(n)
    return out


def _title_sides(title):
    """(respondent-side names, applicant-side names) of an ECHR title, split on the last 'v.'
    and stripped of the procedural parenthetical. The applicant side is returned so the
    learner can reject anything that also occurs there ("AND OTHERS"): a real respondent name
    is a name only one side of the 'v.' ever carries."""
    parts = _V_SPLIT.split(_TITLE_PAREN.sub(" ", title or ""))
    if len(parts) < 2:
        return [], []
    return _names_in(parts[-1]), _names_in(" and ".join(parts[:-1]))


def learn_country_aliases():
    """Build NAME -> CODE once from `records` (47 names on the 2026-08 corpus).

    Only SINGLE-respondent records teach the map: a two-state title ("v. ROMANIA AND
    BULGARIA", respondent "BGR;ROU") cannot say which name is which code. A name is then
    assigned to the code it co-occurs with most often, which discards the few wrong pairings
    that two-state titles still produce ("BULGARIA" appears once under ROU, 49 times
    under BGR).
    """
    global _COUNTRY_ALIASES, _COUNTRY_NAME_RE, _RESPONDENT_CUE_RE, _COUNTRY_CODES
    if _COUNTRY_ALIASES is not None:
        return _COUNTRY_ALIASES
    tally, applicant_side = {}, Counter()
    for r in records:
        if r.get("source") != "echr":
            continue
        resp_names, appl_names = _title_sides(r.get("title"))
        applicant_side.update(appl_names)
        code = (r.get("country") or "").strip().upper()
        if not code or ";" in code:
            continue
        for name in resp_names:
            tally.setdefault(name, Counter())[code] += 1
    alias = {name: c.most_common(1)[0][0] for name, c in tally.items()
             if sum(c.values()) > applicant_side[name]}   # party-name guard, see _title_sides
    for code, names in COUNTRY_ALIASES_EXTRA.items():
        for n in names:
            alias[n.upper()] = code
    _COUNTRY_CODES = {c for r in records
                      for c in (r.get("country") or "").upper().split(";") if c}
    _COUNTRY_ALIASES = alias
    # _NAME_SUFFIX catches the adjectival form the question is as likely to use as the noun:
    # "Romanian courts", "Austrian", and the German endings a DE question carries
    # ("oesterreichischen Recht", "schweizerische Rechtsprechung") -- MEASURED 2026-08-27: with
    # a bare "N?" those two German phrasings matched no state at all and the answer came back
    # 27 RIS/23 Swiss and 43 Swiss/7 RIS respectively, i.e. unrouted. Forms that are not
    # name+suffix (Polish, German for DEU) are still simply not matched.
    names_alt = "|".join(re.escape(n) for n in sorted(alias, key=len, reverse=True))
    _COUNTRY_NAME_RE = re.compile(r"\b(" + names_alt + r")" + _NAME_SUFFIX + r"\b", re.IGNORECASE)
    # "cases AGAINST Austria" / "X v. AUSTRIA" / "gegen Oesterreich": the ECHR reading of a
    # state name, in the Court's own phrasing. Used by route_sources to tell a question about
    # Austrian law from a question about Strasbourg judgments against Austria.
    _RESPONDENT_CUE_RE = re.compile(
        r"\b(?:" + RESPONDENT_CUE + r")\s+(?:the\s+)?(" + names_alt + r")" + _NAME_SUFFIX + r"\b",
        re.IGNORECASE)
    return alias


def scope_countries(query, enabled=None):
    """Respondent-state codes named in the question -> (codes, surface forms).

    Empty set = no scope, i.e. the whole corpus, which is the behaviour for every question
    that names no state. Codes are matched only in upper case (a bare "ROU"); names are
    matched case-insensitively.
    """
    if not (SCOPE_BY_RESPONDENT if enabled is None else enabled):
        return set(), []
    alias = learn_country_aliases()
    codes, seen = set(), []
    for m in _COUNTRY_NAME_RE.finditer(query or ""):
        name = m.group(1).upper()
        codes.add(alias[name])
        if m.group(0) not in seen:
            seen.append(m.group(0))
    for m in re.finditer(r"\b[A-Z]{2,3}\b", query or ""):
        c = m.group(0)
        if c in _COUNTRY_CODES and c not in _CODE_STOPWORDS:
            codes.add(c)
            if c not in seen:
                seen.append(c)
    return codes, seen


def chunk_country_codes(c):
    # ECHR two-state cases carry both ("BGR;ROU"), and both are respondents -- E.S. v. Romania
    # and Bulgaria IS a case against Romania.
    return {x.strip().upper() for x in (c.get("country") or "").split(";") if x.strip()}


def _to_hit(i, score):
    c = chunks[i]
    return {
        "score": float(score), "id": c["doc_id"], "chunk_id": c["chunk_id"],
        "jurisdiction": c["jurisdiction"], "source": c["source"],
        "section": c.get("section", ""), "genre": c.get("genre", ""),
        "country": c["country"], "date": c["date"],
        "url": c["url"], "title": c["title"],
        "snippet": sentence_snippet(c["text"]),   # sentence-aligned display
        "text": c["text"],                        # full chunk for generation
    }


def search_candidates(query, fetch_k=FETCH_K):
    # one FAISS/numpy search; returns the query vector + [(chunk_idx, cosine), ...] desc
    if index is None:
        return None, []
    qv = embed_query(query)
    scores, idxs = index.search(qv, fetch_k)
    cand = [(int(i), float(s)) for s, i in zip(scores[0], idxs[0]) if i >= 0]
    return qv, cand


def mmr_rerank(qv, cand, k=TOP_K, mmr_lambda=MMR_LAMBDA, dup_ceil=MMR_DUP_CEIL):
    # Maximal Marginal Relevance over the candidate set, reusing EMB_MATRIX (no re-embed).
    # next pick = argmax  λ·sim(query) − (1−λ)·max sim(already-picked).
    if not cand:
        return []
    if EMB_MATRIX is None:                         # degrade: plain relevance order
        return cand[:k]
    rel = {i: s for i, s in cand}
    pool = [i for i, _ in cand]
    selected = []
    while pool and len(selected) < k:
        if not selected:
            best = max(pool, key=lambda i: rel[i])
        else:
            sel_mat = EMB_MATRIX[selected]
            def _mmr_score(i):
                div = float(np.max(EMB_MATRIX[i] @ sel_mat.T))   # max sim to picked
                # cruder fallback (toggle): drop near-duplicates outright
                # if div > dup_ceil: return -1e9
                return mmr_lambda * rel[i] - (1.0 - mmr_lambda) * div
            best = max(pool, key=_mmr_score)
        selected.append(best)
        pool.remove(best)
    return [(i, rel[i]) for i in selected]


def retrieve(query, k=TOP_K, fetch_k=FETCH_K, genre_filter=LAW_GENRE_EXCLUDE,
             country_filter=None, source_filter=None, use_mmr=True, mmr_lambda=MMR_LAMBDA,
             score_floor=SCORE_FLOOR, score_margin=SCORE_MARGIN,
             max_per_case=MAX_CHUNKS_PER_CASE, stats=None):
    """Wide, auditable retrieval. Every reduction step is counted into `stats` (pass a dict)
    so an answer can report what was searched, what was dropped and why — a recall claim
    should not rest only on the k things that survived.

    genre_filter: genres to exclude (default {'communicated'}); None/empty = keep all.
    score_margin: keep only candidates within this cosine of the top hit (None = keep all).
                  Relative because e5 cosines occupy a narrow, query-dependent band.
    score_floor : absolute backstop for "nothing in the corpus is on topic" (None = off).
    max_per_case: chunks one document may contribute (None = uncapped); keeps a single long
                  judgment from occupying the whole set, so k chunks span more cases.
    country_filter: respondent codes to keep (None = whole corpus). Applied BEFORE the score
                  cut, because the cut is relative to the top hit: computed on the unfiltered
                  pool, a Norwegian passage sets the bar and every in-scope Romanian one falls
                  below it. With a filter on, the pool is widened (FETCH_K_SCOPED) -- the
                  filter is exact metadata, so recall inside the scope should not be limited
                  by how many out-of-scope chunks happened to outrank it.
    source_filter: corpora to keep ({'ris'}, {'swiss'}, {'echr'}; None = all three). Same
                  before-the-cut placement and for a sharper version of the same reason: the
                  corpora are in different languages, e5 scores same-language passages higher,
                  so an English question cut against an ECHR top hit deletes the German corpus
                  the question was actually about (measured 47/50 ECHR on a Bundesgericht
                  question).
    """
    st = {} if stats is None else stats
    if country_filter or source_filter:
        fetch_k = len(chunks) if FETCH_K_SCOPED is None else max(fetch_k, FETCH_K_SCOPED)
    if country_filter:
        st["scope_countries"] = sorted(country_filter)
    if source_filter:
        st["scope_sources"] = sorted(source_filter)
    qv, cand = search_candidates(query, fetch_k=max(fetch_k, k))
    st["pool_fetched"] = len(cand)
    st["pool_ceiling"] = max(fetch_k, k)
    # True = the index ran out before the ceiling did, i.e. nothing was left unseen
    st["pool_exhausted"] = (len(cand) < st["pool_ceiling"]
                            or st["pool_ceiling"] >= len(chunks))
    # off-topic test on the RAW pool, before any filter: is the candidate list DEEP, or are the
    # few top hits accidents? Measured discriminator — see OFF_TOPIC_DEPTH_* in the config cell.
    depth = cand[OFF_TOPIC_DEPTH_N - 1][1] if len(cand) >= OFF_TOPIC_DEPTH_N else None
    st["depth_rank"] = OFF_TOPIC_DEPTH_N
    st["depth_score"] = round(depth, 4) if depth is not None else None
    st["off_topic"] = depth is not None and depth < OFF_TOPIC_DEPTH_MIN
    if not cand:
        st.update(dropped_genre=0, dropped_out_of_source=0, dropped_out_of_scope=0,
                  dropped_below_cut=0, dropped_case_cap=0, kept=0, distinct_cases=0,
                  score_range=None)
        return []
    if genre_filter:
        gf = set(genre_filter)
        n0 = len(cand)
        cand = [(i, s) for i, s in cand if chunks[i].get("genre") not in gf]
        st["dropped_genre"] = n0 - len(cand)
    else:
        st["dropped_genre"] = 0
    # The absolute floor answers "is this question about the corpus at all?" -- a judgment
    # that must NOT depend on the scope. Inside one respondent State the top cosine is drawn
    # from far fewer documents and sits lower (measured: 0.885 corpus-wide vs 0.837 for ROU on
    # the same question), so keeping the floor after the filter would drop every in-scope
    # passage and report an on-topic question as off-topic. It is therefore applied here, to
    # the unscoped pool, and the in-scope cut below is purely relative.
    if (country_filter or source_filter) and score_floor is not None:
        st["top_cos_unscoped"] = round(max(s for _, s in cand), 4)
        if st["top_cos_unscoped"] < score_floor:
            st.update(dropped_out_of_source=0, dropped_out_of_scope=0,
                      dropped_below_cut=len(cand), dropped_case_cap=0,
                      kept=0, distinct_cases=0, score_range=None, above_cut=0,
                      top_cos=st["top_cos_unscoped"], cut=score_floor,
                      cut_basis="absolute floor (off-topic backstop, judged before the scope)")
            return []
        score_floor = None
    if source_filter:
        sf = set(source_filter)
        n0 = len(cand)
        cand = [(i, s) for i, s in cand if chunks[i]["source"] in sf]
        st["dropped_out_of_source"] = n0 - len(cand)
        st["in_source"] = len(cand)
        if not cand:                       # the corpus is empty for this question, not the topic
            st.update(dropped_out_of_scope=0, dropped_below_cut=0, dropped_case_cap=0, kept=0,
                      distinct_cases=0, score_range=None, top_cos=None, cut=None,
                      cut_basis="none")
            return []
    else:
        st["dropped_out_of_source"] = 0
    if country_filter:
        cf = set(country_filter)
        n0 = len(cand)
        cand = [(i, s) for i, s in cand if chunk_country_codes(chunks[i]) & cf]
        st["dropped_out_of_scope"] = n0 - len(cand)
        st["in_scope"] = len(cand)
        if not cand:                       # scope is empty, not "nothing on topic"
            st.update(dropped_below_cut=0, dropped_case_cap=0, kept=0, distinct_cases=0,
                      score_range=None, top_cos=None, cut=None, cut_basis="none")
            return []
    else:
        st["dropped_out_of_scope"] = 0
    top_cos = max((s for _, s in cand), default=0.0)
    cut = max([x for x in (score_floor,
                           (top_cos - score_margin) if score_margin is not None else None)
               if x is not None], default=None)
    st["top_cos"] = round(top_cos, 4)
    st["cut"] = round(cut, 4) if cut is not None else None
    st["cut_basis"] = ("absolute floor (off-topic backstop)"
                       if cut is not None and score_floor is not None and cut == score_floor
                       else "top hit minus margin" if cut is not None else "none")
    if cut is not None:
        n0 = len(cand)
        cand = [(i, s) for i, s in cand if s >= cut]
        st["dropped_below_cut"] = n0 - len(cand)
    else:
        st["dropped_below_cut"] = 0
    st["above_cut"] = len(cand)
    if not cand:
        st.update(dropped_case_cap=0, kept=0, distinct_cases=0, score_range=None)
        return []
    st["dropped_mmr_pool"] = 0
    if MMR_POOL_CEIL and len(cand) > MMR_POOL_CEIL * k:
        cand = sorted(cand, key=lambda t: -t[1])[:MMR_POOL_CEIL * k]   # best-scoring only
        st["dropped_mmr_pool"] = st["above_cut"] - len(cand)
    # rerank a pool wider than k, so the per-case cap can fall back on the next-best chunk
    ranked = (mmr_rerank(qv, cand, k=min(len(cand), max(k * 3, k)), mmr_lambda=mmr_lambda)
              if use_mmr else cand)
    kept, per_case, capped = [], {}, 0
    for i, sc in ranked:
        doc = chunks[i]["doc_id"]
        if max_per_case and per_case.get(doc, 0) >= max_per_case:
            capped += 1
            continue
        per_case[doc] = per_case.get(doc, 0) + 1
        kept.append((i, sc))
        if len(kept) >= k:
            break
    st["dropped_case_cap"] = capped
    st["kept"] = len(kept)
    st["distinct_cases"] = len(per_case)
    st["score_range"] = [round(kept[-1][1], 4), round(kept[0][1], 4)] if kept else None
    return [_to_hit(i, s) for i, s in kept]


_SOURCE_ROWS = None


def _source_rows():
    """source -> row indices into EMB_MATRIX, built once."""
    global _SOURCE_ROWS
    if _SOURCE_ROWS is None:
        idx = {}
        for i, c in enumerate(chunks):
            idx.setdefault(c["source"], []).append(i)
        _SOURCE_ROWS = {k: np.array(v) for k, v in idx.items()}
    return _SOURCE_ROWS


def respondent_cued(query):
    """Country codes the question names in the ECHR's own respondent phrasing
    ("cases against Austria", "X v. AUSTRIA", "gegen Oesterreich")."""
    alias = learn_country_aliases()
    return {alias[m.group(1).upper()] for m in _RESPONDENT_CUE_RE.finditer(query or "")}


def route_sources(query, enabled=None):
    """Which CORPUS is the question about, and which respondent States inside it?

    Returns {"sources": [...] | None, "countries": {codes}, "matched": [surface forms],
             "basis": [one readable reason per decision]}.
    `sources` None = the whole corpus (the question names no legal system); one entry = a hard
    source filter; 2+ = a comparison, which retrieves once per corpus (retrieve_per_source).

    The two readings of a country name are separated mechanically, never by intent-guessing:
      * a state we hold a national corpus for goes to THAT corpus ("in Austria" -> RIS)
      * unless the question phrases it as a respondent ("against Austria" -> ECHR, AUT)
      * a state we hold no corpus for can only be an ECHR respondent ("against Romania")
    A question naming both the ECHR and a national system stays a 2-source comparison, so no
    reading is silently dropped.
    """
    if not (SOURCE_ROUTING if enabled is None else enabled):
        codes, names = scope_countries(query)     # legacy: a state name is a respondent filter
        return {"sources": None, "countries": codes, "matched": names,
                "basis": ["source routing off"]}
    # enabled=True: a state name is read here even when respondent scoping is off, because
    # WHICH CORPUS answers and WHETHER it is filtered by respondent State are two decisions
    codes, names = scope_countries(query, enabled=True)
    sources = {src for src, pat in SOURCE_PATTERNS.items()
               if re.search(pat, query or "", re.IGNORECASE)}
    basis = [f"names {SOURCE_LABELS[s]}" for s in sorted(sources)]
    cued, respondent = respondent_cued(query), set()
    for c in sorted(codes):
        national = NATIONAL_CORPUS.get(c)
        if national and c not in cued:
            sources.add(national)
            basis.append(f"'{c}' as a legal system -> {national} corpus")
        else:
            sources.add("echr")
            if SCOPE_BY_RESPONDENT:               # the filter is the other decision
                respondent.add(ECHR_RESPONDENT_EQUIV.get(c, c))
            basis.append(f"'{c}' as an ECHR respondent State "
                         f"({'respondent cue' if c in cued else 'no corpus of its own'})")
    # a respondent filter is meaningful only inside the ECHR corpus; a national corpus IS the
    # state, so filtering it by respondent code would be a second, redundant restriction
    return {"sources": sorted(sources) or None, "countries": respondent, "matched": names,
            "basis": basis or ["no legal system named -- whole corpus"]}


def retrieve_per_source(query, sources, k_per_source=None, genre_filter=LAW_GENRE_EXCLUDE,
                        score_margin=SCORE_MARGIN, max_per_case=MAX_CHUNKS_PER_CASE,
                        mmr_lambda=MMR_LAMBDA, stats=None):
    """One search PER corpus, each with its own relative cut, merged round-robin.

    The absolute SCORE_FLOOR is deliberately NOT applied here: it was calibrated on English
    on-topic scores, and a cross-lingual match (English question, German judgment) sits lower
    by construction. Applying it would delete exactly the passages a comparison needs. Each
    source's per-source top cosine is recorded instead, so the cross-lingual gap is visible
    rather than silently filtered away.

    Round-robin merge (echr1, ris1, swiss1, echr2, ...) matters: the prompt is packed in list
    order, so interleaving is what puts every jurisdiction inside the generator's window.
    """
    st = {} if stats is None else stats
    st["mode"] = "per_source"
    st["sources_requested"] = list(sources)
    st["per_source"] = {}
    k_each = k_per_source or max(1, TOP_K // max(1, len(sources)))
    qv = embed_query(query)
    rows_by_src = _source_rows()
    picked_by_src = {}
    for src in sources:
        rows = rows_by_src.get(src)
        if rows is None or not len(rows):
            st["per_source"][src] = {"in_corpus": 0, "kept": 0}
            continue
        sims = (qv @ EMB_MATRIX[rows].T)[0]
        top = np.argsort(-sims)[:COMPARATIVE_FETCH]
        cand = [(int(rows[j]), float(sims[j])) for j in top]
        entry = {"in_corpus": int(len(rows)), "scored": len(cand),
                 "top_cos": round(cand[0][1], 4) if cand else None}
        if genre_filter:
            gf = set(genre_filter)
            cand = [(i, sc) for i, sc in cand if chunks[i].get("genre") not in gf]
        if score_margin is not None and cand:
            cut = cand[0][1] - score_margin
            entry["cut"] = round(cut, 4)
            cand = [(i, sc) for i, sc in cand if sc >= cut]
        ranked = mmr_rerank(qv, cand, k=min(len(cand), k_each * 3), mmr_lambda=mmr_lambda)
        kept, per_case = [], {}
        for i, sc in ranked:
            doc = chunks[i]["doc_id"]
            if max_per_case and per_case.get(doc, 0) >= max_per_case:
                continue
            per_case[doc] = per_case.get(doc, 0) + 1
            kept.append((i, sc))
            if len(kept) >= k_each:
                break
        entry["kept"] = len(kept)
        entry["cases"] = len(per_case)
        st["per_source"][src] = entry
        picked_by_src[src] = kept
    merged = []
    for rank in range(max((len(v) for v in picked_by_src.values()), default=0)):
        for src in sources:                       # round-robin keeps every jurisdiction early
            if src in picked_by_src and rank < len(picked_by_src[src]):
                merged.append(picked_by_src[src][rank])
    st["kept"] = len(merged)
    st["distinct_cases"] = len({chunks[i]["doc_id"] for i, _ in merged})
    st["score_range"] = ([round(min(s for _, s in merged), 4),
                          round(max(s for _, s in merged), 4)] if merged else None)
    return [_to_hit(i, s) for i, s in merged]


print(f"retrieve() ready — genre filter + source route + respondent scope + cut at "
      f"top-{SCORE_MARGIN} (floor {SCORE_FLOOR}) + MMR + max {MAX_CHUNKS_PER_CASE}/case, "
      f"counted at every step")
print(f"route_sources() ready — corpora {sorted(SOURCE_PATTERNS)}; a country name routes to "
      f"its own corpus {NATIONAL_CORPUS}, or to ECHR respondent scope under a respondent cue")

retrieve() ready — genre filter + MMR


### 6b. Routing check — the two readings of a country name
The route decides which corpus may answer, so it is checked the same way any other classifier
is: a fixed battery with the expected corpus **and** the expected respondent scope written
down next to each question. The interesting rows are the pairs — `in Austria` vs
`against Austria`, `in der Schweiz` vs `cases against Switzerland` — where the same state name
must reach two different bodies of documents. German adjectival forms are in the battery
because they were the measured failure (`oesterreichischen`, `schweizerische` matched no state
at all before `_NAME_SUFFIX`).

In [ ]:
# expected: (corpora the question may be answered from, respondent codes inside the ECHR)
ROUTE_CASES = [
    # a legal system named -> that corpus, no respondent filter
    ("How is the best interest of the child formulated in Austria?",       ["ris"],   []),
    ("Wie wird das Kindeswohl im österreichischen Recht formuliert?",      ["ris"],   []),
    ("How does the OGH scope parental alienation?",                        ["ris"],   []),
    ("Wie formuliert die schweizerische Rechtsprechung das Kindeswohl?",   ["swiss"], []),
    ("How does the Bundesgericht formulate the best interest of the child?", ["swiss"], []),
    ("How is contact enforcement scoped in Swiss case law?",               ["swiss"], []),
    ("Wie formuliert der EGMR das Kindeswohl?",                            ["echr"],  []),
    ("What does the Strasbourg court say about contact rights?",           ["echr"],  []),
    # the SAME state names, in the Court's respondent phrasing -> ECHR + a scope
    ("How is parental alienation formulated in ECHR cases against Austria?", ["echr"], ["AUT"]),
    ("cases against Switzerland about contact rights",                     ["echr"],  ["CHE"]),
    ("Welche Rolle spielt der Kindeswille in Entscheidungen gegen Österreich?", ["echr"], ["AUT"]),
    # a state we hold no national corpus for can only be an ECHR respondent
    ("How is parental alienation treated in Germany?",                     ["echr"],  ["DEU"]),
    ("What does the Court say in cases against Romania about contact rights?", ["echr"], ["ROU"]),
    # 2+ systems named -> comparison (retrieve_per_source), not a single filter
    ("How do the ECHR, the OGH and Swiss courts each formulate the best interest of the child?",
     ["echr", "ris", "swiss"], []),
    ("Compare Austria and Switzerland on the enforcement of contact rights",
     ["ris", "swiss"], []),
    # none named -> the whole corpus, which is what an unrouted question always got
    ("When may contact be restricted against the child's expressed wishes?", [],       []),
]

_route_fails = 0
for _q, _exp_src, _exp_resp in ROUTE_CASES:
    _r = route_sources(_q)
    _got = (_r["sources"] or [], sorted(_r["countries"]))
    if _got != (_exp_src, _exp_resp):
        _route_fails += 1
        print(f"  FAIL {_q[:64]!r}\n       expected {(_exp_src, _exp_resp)} got {_got}")
print(f"routing check: {len(ROUTE_CASES) - _route_fails}/{len(ROUTE_CASES)} questions routed "
      f"as written ({_route_fails} failures)")
print("  'against' as a respondent cue is safe on the phrase it looks most like: "
      f"'restricted against the child\'s expressed wishes' -> "
      f"{sorted(respondent_cued(ROUTE_CASES[-1][0])) or 'no state'}")

## 7. Abstention — Layer 1 (out-of-paradigm), deterministic
Catches aggregate/counting/statistical questions before generation; returns an abstention
plus the relevant cases. Explicit `abstained`/`abstain_type` fields. Biased to over-abstain.

In [ ]:
AGGREGATE_PATTERNS = [
    r"\bhow many\b", r"\bhow much\b", r"\bhow often\b",
    r"\bwhat (?:proportion|percentage|share|fraction|number)\b",
    r"\bmost (?:cases|courts|judgments|decisions|of)\b", r"\bmajority of\b",
    r"\bon average\b", r"\baverage\b", r"\bpercentage\b", r"\bproportion\b",
    r"\bnumber of\b", r"\bcount of\b", r"\bfrequenc", r"\brate of\b",
    r"\btrend\b", r"\bover the years\b", r"\bstatistic", r"\btypically\b",
    r"\busually\b", r"\bin general\b",
    r"\bwie viele\b", r"\bwie häufig\b", r"\bwie oft\b", r"\banteil\b",
    r"\bdurchschnitt", r"\bprozent\b", r"\bmehrheit\b", r"\bdie meisten\b",
    r"\bin der regel\b", r"\bstatistik", r"\bhäufigkeit\b", r"\btendenz\b",
    r"\bquote\b", r"\bzahl der\b", r"\binsgesamt\b", r"\bgesamtzahl\b",
]
_AGG_RE = re.compile("|".join(AGGREGATE_PATTERNS), re.IGNORECASE)


def aggregate_trigger(query):
    m = _AGG_RE.search(query or "")
    return m.group(0) if m else None


ABSTAIN_MSG_AGGREGATE = (
    "Abstaining: this is an aggregate/statistical question (counts, proportions, averages, "
    "trends). This tool reads a handful of individual passages and cannot compute corpus-wide "
    "statistics without fabricating them. The most relevant individual cases are listed below.")
NOT_FOUND = "Not found in the corpus"

print(f"Type-1 detector ready ({len(AGGREGATE_PATTERNS)} patterns)")

Type-1 detector ready (36 patterns)


## 8. Grounded generation (Ollama optional) + Layer-2/4 abstention
Layer 2 enforced in the prompt (reply exactly "Not found in the corpus"); Layer 4 is
deterministic — the topic is in the corpus but the requested **scope** is empty, which is a
different statement from "not found". Degrades to retrieval-only if Ollama is absent.

`GEN_ANSWER_STYLE` sets the answer's **shape**, not its evidence: `enumerate` walks the
retrieved list one source per paragraph (the unconstrained default); `synthesis` states each
proposition once and hangs every supporting id off it. Same sources, same citation duty —
what changes is whether the reader gets a pile of case summaries or one picture. A rule also
guards the commonest misattribution: a case name **quoted inside** a source passage is that
judgment citing another judgment, not a retrieved source of ours.

In [ ]:
OLLAMA_OK = False
if USE_LLM:
    try:
        import ollama
        OLLAMA_OK = True
    except Exception as e:
        print(f"Ollama unavailable ({e}) — retrieval-only mode")

SYSTEM_PROMPT = (
    "You are a legal retrieval assistant over a multi-jurisdiction corpus "
    "(ECHR, Austrian OGH, Swiss courts). Answer ONLY from the numbered SOURCES provided.\n"
    "1. Use no outside knowledge. If the sources do not support an answer, reply "
    "EXACTLY: Not found in the corpus\n"
    "2. For every statement, name the jurisdiction (ECHR, AT (OGH) or CH) and cite the source "
    "id in square brackets, e.g. [echr:001-157293:2]. COPY the id character by character from "
    "an 'id=' header above. Never assemble, guess or adapt one: an id that is not in the "
    "sources is a fabricated reference even when the sentence it supports is true.\n"
    "3. Never attribute one jurisdiction's rule to another; never merge jurisdictions.\n"
    "4. Some sources are EXTRACTS: '...' marks omitted text. Never assume the text either "
    "side of a '...' is continuous, and never complete a sentence that the extract cuts.\n"
    "5. Mind the section label. FACTS and PROCEDURE passages often restate a party's "
    "submission or a domestic court's ruling -- report those as such, not as the Court's own "
    "holding. Only LAW/OPERATIVE passages state what the Court itself held.\n"
    "6. A case name appearing INSIDE a source passage (e.g. 'as the Court held in Maumousseau "
    "and Washington v. France') is that judgment citing another judgment. It is NOT one of "
    "your sources. Attribute the point to the id of the passage you actually read; never "
    "present a name quoted inside a passage as a separate retrieved case.\n"
    "7. Be concise and factual.")

# Answer SHAPE, appended to the shared rules. Only the ORGANISATION differs -- both styles
# answer from the same sources under the same citation duty. "enumerate" reproduces the
# default behaviour (kept so the two can be compared in the faithfulness evaluation);
# "synthesis" states each proposition once and hangs every supporting id off it, which is
# what turns a walk through the retrieved list into an answer.
ANSWER_STYLE_BLOCKS = {
    "enumerate": (
        "8. Go through the sources that bear on the question, one short paragraph each."),
    "synthesis": (
        "8. ORGANISE BY PROPOSITION, NOT BY CASE. Group the sources by what they SAY: state "
        "each proposition once, in your own words, and put every id that supports it right "
        "after it, e.g. '... must be a primary consideration [a][b][c]'. Never write one "
        "bullet per case, and never begin a sentence with 'In the case of ...'.\n"
        "9. Put the most widely supported proposition first. If only one source supports a "
        "proposition, write 'only one source states this'. If sources pull in different "
        "directions, say so instead of averaging them.\n"
        "10. Do not add a closing disclaimer about completeness or representativeness: the "
        "system reports its own scope and grounding counts separately."),
}

# shape instruction in the USER turn as well -- a 3B model follows a concrete template far
# more reliably than a numbered rule twenty lines up in the system prompt
ANSWER_STYLE_TASK = {
    "enumerate": "Answer using only the sources above, with jurisdiction + [id] citations.",
    "synthesis": (
        "Answer using only the sources above. Write 3-6 propositions, most-supported first, "
        "each on its own line as:\n"
        "- <the proposition, one or two sentences, in your own words> [id][id]\n"
        "State each proposition once. Do not walk through the sources one by one, and do not "
        "summarise cases individually."),
}


def system_prompt(style=None):
    style = GEN_ANSWER_STYLE if style is None else style
    return SYSTEM_PROMPT + "\n" + ANSWER_STYLE_BLOCKS[style]


def _prompt_block(n, h, query, form):
    """One numbered SOURCE. The header carries section + genre because an extract strips the
    surrounding cues a reader would otherwise use to tell a holding from a submission."""
    sec = f" | section={h['section']}" if h.get("section") else ""
    gen = f" | genre={h['genre']}" if h.get("genre") else ""
    head = (f"[{n}] id={h['chunk_id']} | jurisdiction={h['jurisdiction']}{sec}{gen} | "
            f"title={h['title']}")
    if form == "full":
        return f"{head}\n{h['text']}"
    wins, basis = kwic_extracts(h["text"], query, window=GEN_EXTRACT_WINDOW,
                                max_windows=GEN_EXTRACT_WINDOWS)
    tag = {"query_anchor": "EXTRACT",
           "semantic_anchor": "EXTRACT (passage closest to the question — no shared wording, "
                              "which is normal across languages)",
           }.get(basis, "EXTRACT (opening of the passage — it contains no query term)")
    return f"{head} | {tag}\n" + " ... ".join(w["extract"] for w in wins)


def build_context(hits, query, char_budget=None, packing=None, full_head=None):
    """Pack hits into the prompt until the budget is spent, and report WHICH hits fit and in
    WHAT form. Retrieval is wide on purpose; the generator's window is not. Returning `used`
    and `forms` keeps that honest — a hit that never entered the prompt cannot have grounded
    anything, and a hit that entered as an extract grounded only the sentences it carried.

    packing: "full" = whole chunks (~6 sources fit); "extract" = anchored windows only;
             "hybrid" = the top `full_head` sources full, the long tail as extracts, which is
             what buys case breadth without stripping context from the likeliest quotes.
             None = read the current GEN_PACKING, so the setting can be flipped between
             questions (e.g. to A/B packing against the faithfulness evaluation).
    """
    # resolved per CALL, not per definition: a default argument would freeze the value at
    # cell-execution time, so setting GEN_PACKING="full" mid-session would silently do nothing.
    char_budget = GEN_CHAR_BUDGET if char_budget is None else char_budget
    packing = GEN_PACKING if packing is None else packing
    full_head = GEN_FULL_HEAD if full_head is None else full_head
    blocks, used, forms, n_chars = [], [], {}, 0
    for h in hits:
        if len(used) >= GEN_MAX_SOURCES:      # the cliff is a source COUNT, not a char budget
            break
        form = ("full" if packing == "full" else
                "extract" if packing == "extract" else
                "full" if len(used) < full_head else "extract")
        block = _prompt_block(len(used) + 1, h, query, form)
        if used and n_chars + len(block) > char_budget and form == "full" and packing != "full":
            form = "extract"                       # a full block may not fit where a window does
            block = _prompt_block(len(used) + 1, h, query, form)
        if used and n_chars + len(block) > char_budget:
            continue                               # skip, don't stop: a later source may fit
        blocks.append(block)
        used.append(h)
        forms[h["chunk_id"]] = form
        n_chars += len(block)
    return "\n\n".join(blocks), used, forms


def comparative_instruction(used):
    """Force a per-jurisdiction structure when the question compares jurisdictions.

    Measured twice: without it the model answers a three-jurisdiction question from whichever
    passages read most fluently (all ECHR), and in one run it relabelled ECHR ids as Swiss to
    fill the gap. Naming the jurisdictions actually present — with their source counts — makes
    a missing one a fact the model must state, not a hole it improvises around.
    """
    counts = {}
    for h in used:
        counts[h["jurisdiction"]] = counts.get(h["jurisdiction"], 0) + 1
    if len(counts) < 2:
        return ""
    listing = ", ".join(f"{j} ({n} source{'s' if n > 1 else ''})" for j, n in counts.items())
    return ("\n\nThe SOURCES cover these jurisdictions: " + listing + ". Write ONE SECTION PER "
            "JURISDICTION, headed by the jurisdiction name, in that order. Inside a section "
            "cite ONLY ids from that jurisdiction — the id prefix says which (echr: = ECHR, "
            "ris: = AT (OGH), swiss: = CH). If a jurisdiction's sources do not answer the "
            "question, write exactly: Not found in the corpus.")


def generate(query, hits, char_budget=None, style=None, scope_note="", comparative=False):
    """Returns (answer, hits_in_the_prompt, {chunk_id: form}, prompt_chars).

    scope_note states a metadata restriction that retrieval has ALREADY enforced (e.g. every
    source is a case against Romania). Saying so stops the model from hedging about states it
    cannot see -- it is a description of the source set, never an instruction to filter.
    """
    if not OLLAMA_OK or not hits:
        return None, [], {}, 0
    style = GEN_ANSWER_STYLE if style is None else style
    context, used, forms = build_context(hits, query, char_budget=char_budget)
    user = (f"{scope_note}SOURCES:\n{context}\n\nQUESTION: {query}\n\n"
            + ANSWER_STYLE_TASK[style]
            + (comparative_instruction(used) if comparative else ""))
    try:
        # num_ctx is explicit: ollama's 4096 default silently truncates a longer prompt,
        # which would drop sources the model is simultaneously instructed to cite.
        resp = ollama.chat(model=GEN_MODEL, messages=[
            {"role": "system", "content": system_prompt(style)},
            {"role": "user", "content": user}],
            options={"temperature": 0, "num_ctx": GEN_NUM_CTX})
        return resp["message"]["content"].strip(), used, forms, len(context)
    except Exception as e:
        return f"[generation unavailable: {e}]", used, forms, len(context)


# Layer-3 (genre) abstention message — surfaced alongside the communicated cases as context.
ABSTAIN_MSG_BOILERPLATE = (
    "Abstaining: pending applications on this topic exist in the corpus (ECHR communicated "
    "cases — listed separately below as CONTEXT), but no decided ruling states the principle, "
    "so the law cannot be provided. Communicated cases restate the dispute and pose questions; "
    "they do not hold.")


# Layer-4 abstention: the topic is in the corpus but the requested SCOPE is not. Distinct
# from "not found": the honest report is that this respondent State has no such passage, not
# that the corpus is silent on the topic.
ABSTAIN_MSG_SOURCE = (
    "Abstaining: nothing in {source} is on this topic. Passages exist in the other corpora; "
    "answering from those would silently change the question -- a question about one legal "
    "system cannot be answered out of another one's case law.")


ABSTAIN_MSG_SCOPE = (
    "Abstaining: no passage in the corpus is both on this topic and within the requested "
    "scope ({scope}). Passages on the topic exist for other respondent States; answering "
    "from those would silently change the question.")


def _communicated_context(query, genre_filter, k, country_filter=None,
                          source_filter=None):
    # Pull the genres we just excluded (the boilerplate) so they can be shown as context,
    # never as the answer basis. Same vectors, separate MMR pass.
    if not genre_filter:
        return []
    if source_filter and "echr" not in set(source_filter):
        return []          # 'communicated' is an ECHR genre; outside it there is no such context
    qv, cand = search_candidates(query, fetch_k=FETCH_K)
    if not cand:
        return []
    # same similarity cut as retrieve(): without it an off-topic question would come back as
    # "pending applications exist on this topic" (Layer 3) instead of "not found" (Layer 2).
    cut = max(s for _, s in cand) - SCORE_MARGIN   # off-topic is handled upstream by the
                                                  # depth test, not by an absolute floor
    keep = set(genre_filter)
    comm = [(i, s) for i, s in cand if chunks[i].get("genre") in keep and s >= cut]
    if country_filter:                      # context must obey the same scope as the answer
        cf = set(country_filter)
        comm = [(i, s) for i, s in comm if chunk_country_codes(chunks[i]) & cf]
    return [_to_hit(i, s) for i, s in mmr_rerank(qv, comm, k=k)]


def answer(query, k=TOP_K, genre_filter=LAW_GENRE_EXCLUDE, scope=None, sources=None,
           style=None):
    """scope:   None = read the question (SCOPE_BY_RESPONDENT); a set of codes = force it;
                an empty set / False = search the whole corpus whatever the question names.
    sources:    None = read the question (route_sources); a list/set = force the corpora;
                an empty set / False = search all three whatever the question names."""
    route = route_sources(query)
    codes, names = ((route["countries"], route["matched"]) if scope is None
                    else (set(scope or ()), sorted(scope or ())))
    srcs = (route["sources"] if sources is None else (sorted(sources) or None))
    result = {"query": query, "abstained": False, "abstain_type": None,
              "abstain_reason": None, "hits": [], "context": [], "answer": None,
              "used": [], "prompt_forms": {}, "retrieval_stats": {}, "grounding": {},
              "comparative_sources": None,
              "route": {"sources": srcs, "basis": route["basis"],
                        "source": "question" if sources is None else "explicit"},
              "scope": {"countries": sorted(codes), "matched": names,
                        "source": "question" if scope is None else "explicit"},
              "answer_style": GEN_ANSWER_STYLE if style is None else style,
              "mode": "generate" if OLLAMA_OK else "retrieval_only"}
    # substantive retrieval: communicated routed out, source + scope enforced, MMR-diversified
    stats = {}
    comparative = srcs if (COMPARATIVE_ON and srcs and len(srcs) >= 2) else None
    src_filter = None if comparative else srcs
    if comparative:
        # a question naming 2+ jurisdictions must search each corpus separately, or it is
        # answered from whichever corpus shares the question's language (measured: 50/50 ECHR)
        hits = retrieve_per_source(query, comparative, genre_filter=genre_filter, stats=stats)
    else:
        # exactly one corpus named -> the same failure at k=1 jurisdiction, fixed by a hard
        # source filter rather than by hoping the embedding separates the corpora
        hits = retrieve(query, k=k, genre_filter=genre_filter, country_filter=codes,
                        source_filter=src_filter, stats=stats)
    result["comparative_sources"] = comparative
    result["hits"] = hits
    result["retrieval_stats"] = stats
    # the excluded boilerplate, surfaced separately as inspectable context
    result["context"] = _communicated_context(query, genre_filter, COMMUNICATED_CTX_K,
                                              country_filter=codes, source_filter=src_filter)

    # Layer 1 — out-of-paradigm aggregate question
    trig = aggregate_trigger(query)
    if trig:
        result.update(abstained=True, abstain_type="type1_out_of_paradigm",
                      abstain_reason=f"aggregate pattern: '{trig}'", answer=ABSTAIN_MSG_AGGREGATE)
        return result
    # if hits and hits[0]["score"] < SCORE_FLOOR:
    #     result.update(abstained=True, abstain_type="type2_low_score", answer=NOT_FOUND); return result

    # Layer 4b — the routed SOURCE, not the topic, is empty. Reported before the respondent
    # scope because it is the coarser restriction of the two.
    if not hits and src_filter and stats.get("dropped_out_of_source"):
        result.update(abstained=True, abstain_type="type4_source_empty",
                      abstain_reason=f"no on-topic passage in {src_filter}",
                      answer=ABSTAIN_MSG_SOURCE.format(
                          source=", ".join(SOURCE_LABELS.get(s, s) for s in src_filter)))
        return result

    # Layer 4 — the scope, not the topic, is empty.
    if not hits and codes and stats.get("dropped_out_of_scope"):
        scope_txt = ", ".join(names or sorted(codes))
        result.update(abstained=True, abstain_type="type4_scope_empty",
                      abstain_reason=f"no in-scope passage for {sorted(codes)}",
                      answer=ABSTAIN_MSG_SCOPE.format(scope=scope_txt))
        return result

    # Layer 2b — the question is not about this corpus at all (depth test, not peak score).
    if stats.get("off_topic"):
        result.update(abstained=True, abstain_type="type2_off_topic", context=[],
                      abstain_reason=f"shallow candidate list: rank {stats.get('depth_rank')} "
                                     f"scores {stats.get('depth_score')} < {OFF_TOPIC_DEPTH_MIN}",
                      answer=NOT_FOUND)
        return result

    # Layer 2c — on topic, but the cut left nothing. Distinct from Layer 3 on purpose: saying
    # "no decided ruling states the principle" when passages merely fell below a threshold is a
    # claim about the CORPUS derived from a fact about the THRESHOLD. Measured: that message
    # fired on a contact-restriction question whose best judgment missed the cut by 0.003.
    if not hits and stats.get("above_cut") == 0 and stats.get("pool_fetched"):
        result.update(abstained=True, abstain_type="type2_below_cut",
                      abstain_reason=f"no passage cleared the cut "
                                     f"(best {stats.get('top_cos')} vs cut {stats.get('cut')})",
                      answer=NOT_FOUND)
        return result

    # Layer 3 — genre/boilerplate: nothing substantive survived, only communicated boilerplate.
    if not hits:
        if result["context"]:
            result.update(abstained=True, abstain_type="type3_boilerplate_only",
                          abstain_reason="boilerplate_only", answer=ABSTAIN_MSG_BOILERPLATE)
        else:
            result.update(abstained=True, abstain_type="type2_no_hits", answer=NOT_FOUND)
        return result

    scope_note = ""
    if src_filter:
        scope_note += (f"SOURCE: every passage below comes from "
                       f"{', '.join(SOURCE_LABELS.get(s, s) for s in src_filter)}, because "
                       f"that is the legal system the question names. The other corpora were "
                       f"filtered out before you saw them: answer for this one, and do not "
                       f"attribute anything below to another court.\n\n")
    if codes:
        scope_note += (f"SCOPE: every source below is a case whose respondent State is "
                       f"{', '.join(names or sorted(codes))} ({'/'.join(sorted(codes))}). "
                       f"Cases against other States were filtered out before you saw them, so "
                       f"answer for this State and do not hedge about the others.\n\n")
    (result["answer"], result["used"], result["prompt_forms"],
     result["prompt_chars"]) = generate(query, hits, style=style, scope_note=scope_note,
                                       comparative=bool(comparative))
    result["grounding"] = grounding_report(result)
    return result


print(f"answer() ready — mode: {'generate' if OLLAMA_OK else 'retrieval_only'} | "
      f"style: {GEN_ANSWER_STYLE} | routing: source={SOURCE_ROUTING} "
      f"respondent={SCOPE_BY_RESPONDENT} | abstention layers: 1 aggregate · 2 no-hits · "
      "3 boilerplate_only · 4 scope_empty · 4b source_empty")

answer() ready — mode: generate | abstention layers: 1 aggregate · 2 no-hits · 3 boilerplate_only


## 7. Retrieval provenance — recall, grounding, KWIC extracts

Every grounded answer writes a JSON audit next to it (`reports/retrieval_provenance/`).
It separates three counts that are easy to conflate:

| count | meaning |
|---|---|
| **retrieved** | passages above `SCORE_FLOOR` kept by MMR + the per-case cap — the recall set |
| **in prompt** | the subset that fit `GEN_CHAR_BUDGET`; the rest could not have grounded anything |
| **cited** | chunk ids the answer text actually names |

Case counts are de-duplicated: 50 chunks are not 50 judgments. Each retrieved chunk carries
KWIC extracts (±200 chars around a query term). Retrieval is semantic, so a chunk may share
no word with the question — that is labelled `no_lexical_anchor` rather than dressed up as a
keyword hit.

In [ ]:
import json as _json_prov
import re as _re_prov
from datetime import datetime as _dt_prov

# Bracket-AGNOSTIC and case-insensitive on purpose. The model's citation style drifts
# ([echr:001-175359:8] -> (ECHR:001-175359:8)), and a syntax-anchored parser then reports
# "0 cases cited" for an answer full of citations -- a false zero is worse than no audit.
CITATION_RE = _re_prov.compile(r"\b([A-Za-z_]{2,12}:[\w.\-/]+:\d+)")

_ALL_CHUNK_IDS = None


def _all_chunk_ids():
    global _ALL_CHUNK_IDS
    if _ALL_CHUNK_IDS is None:
        _ALL_CHUNK_IDS = {c["chunk_id"].lower() for c in chunks}
    return _ALL_CHUNK_IDS
_SENT_BOUND = _re_prov.compile(r"[.!?]\s+")


def _snap_to_sentences(text, lo, hi, max_grow=220):
    """Grow a window out to the nearest sentence boundaries. A window that starts mid-clause
    loses the subject and any preceding negation — which is how an extract comes to say the
    opposite of its source."""
    left = [m.end() for m in _SENT_BOUND.finditer(text, max(0, lo - max_grow), lo)]
    if left:
        lo = left[-1]
    right = _SENT_BOUND.search(text, hi, min(len(text), hi + max_grow))
    if right:
        hi = right.start() + 1
    return lo, hi


def _case_key(h):
    return f"{h['source']}:{h['id']}"          # source-qualified case identity


def semantic_windows(text, query, window=KWIC_WINDOW, max_windows=1, stride=2):
    """Pick the passage of `text` closest to the question by EMBEDDING, not by keyword.

    This is what makes a cross-lingual corpus usable. An English question shares no word with
    a German judgment, so lexical anchoring finds nothing and the fallback used to hand the
    model the OPENING of the chunk -- which in an OGH decision is admissibility boilerplate
    ("Durchbrechung des Neuerungsverbots"), not the reasoning that got the chunk retrieved.
    The model then ignored every German source, correctly, because it was given noise.

    Same multilingual model that retrieved the chunk, so the window it picks is the one that
    made the chunk rank in the first place.
    """
    sents = [x for x in _SENT_BOUND.split(text) if x.strip()]
    if not sents:
        return []
    spans, pos = [], 0
    starts = []
    for x in sents:                                  # char offset of every sentence
        starts.append(text.find(x, pos))
        pos = starts[-1] + len(x)
    per = max(1, window * 2 // max(1, sum(len(x) for x in sents) // len(sents)))
    for i in range(0, len(sents), max(1, stride)):
        block = " ".join(sents[i:i + per])
        if len(block) < 40:
            continue
        lo = starts[i]
        spans.append((lo, min(len(text), lo + len(block)), block))
    if not spans:
        return []
    qv = embed_query(query)
    mat = embed_passages([b for _, _, b in spans], progress=False)
    sims = (qv @ mat.T)[0]
    best = np.argsort(-sims)[:max_windows]
    return [{"basis": "semantic_anchor", "anchor": None, "cos": round(float(sims[j]), 4),
             "char_range": [int(spans[j][0]), int(spans[j][1])]} for j in sorted(best)]


def kwic_extracts(text, query, window=KWIC_WINDOW, max_windows=KWIC_MAX_PER_CHUNK, snap=True,
                  semantic=True):
    """Query-anchored KWIC: +/-`window` chars around each query term found in the chunk.

    Retrieval here is SEMANTIC, so a genuinely relevant chunk may share no word with the
    question -- always true across languages. With `semantic=True` that case falls back to
    embedding-picked windows (`semantic_anchor`); with False it degrades to the head of the
    chunk (`no_lexical_anchor`). Neither is ever presented as a keyword match that did not
    happen.
    """
    # WORD-boundary matching: substring matching made "best" fire inside "besteht" and "each"
    # inside "beachten", pointing German extracts at arbitrary windows and mislabelling them
    # as query anchors.
    terms = sorted({w.lower() for w in _re_prov.findall(r"\w{4,}", query, _re_prov.UNICODE)},
                   key=len, reverse=True)
    low = text.lower()
    spans = []
    for t in terms:
        for m in _re_prov.finditer(rf"\b{_re_prov.escape(t)}\b", low):
            spans.append((m.start(), m.end(), t))
            if len(spans) >= max_windows * 4:
                break
    if not spans:
        if semantic:
            wins = semantic_windows(text, query, window=window, max_windows=max_windows)
            if wins:
                for w in wins:
                    lo, hi = w["char_range"]
                    if snap:
                        lo, hi = _snap_to_sentences(text, lo, hi)
                        w["char_range"] = [lo, hi]
                    w["extract"] = (("..." if lo > 0 else "") + text[lo:hi].strip()
                                    + ("..." if hi < len(text) else ""))
                return wins, "semantic_anchor"
        head = text[:window * 2].strip()
        return [{"basis": "no_lexical_anchor", "anchor": None, "char_range": [0, len(head)],
                 "extract": head + ("..." if len(text) > len(head) else "")}], "no_lexical_anchor"
    spans.sort()
    out = []
    for a, b, term in spans:
        lo, hi = max(0, a - window), min(len(text), b + window)
        if snap:
            lo, hi = _snap_to_sentences(text, lo, hi)
        if out and lo <= out[-1]["char_range"][1]:          # merge overlapping windows
            out[-1]["char_range"][1] = max(out[-1]["char_range"][1], hi)
            if term not in out[-1]["anchor"]:
                out[-1]["anchor"] += f", {term}"
            continue
        out.append({"basis": "query_anchor", "anchor": term, "char_range": [lo, hi]})
        if len(out) >= max_windows:
            break
    for w in out:
        lo, hi = w["char_range"]
        w["extract"] = ("..." if lo > 0 else "") + text[lo:hi].strip() + \
                       ("..." if hi < len(text) else "")
    return out, "query_anchor"


_JURIS_LABEL = _re_prov.compile(
    r"\b(ECHR|ECtHR|Strasbourg|OGH|Austrian|Austria|CH|Swiss|Switzerland)\b",
    _re_prov.IGNORECASE)
_JURIS_CANON = {"echr": "ECHR", "ecthr": "ECHR", "strasbourg": "ECHR",
                "ogh": "AT (OGH)", "austrian": "AT (OGH)", "austria": "AT (OGH)",
                "ch": "CH", "swiss": "CH", "switzerland": "CH"}


# The model also cites by PROMPT POSITION ("[1]", "[15]") instead of by id -- measured in the
# per-jurisdiction answer format. Those are resolvable: build_context numbers sources in `used`
# order, so [n] IS used[n-1]. Left unresolved they would count as zero citations and hide
# whatever they were attached to.
POSITIONAL_RE = _re_prov.compile(r"\[(\d{1,2})\]")


def resolve_citations(answer_text, used):
    """Every citation in the answer as (span_end, hit, kind). kind='id' when the answer names
    the chunk id, 'position' when it names the prompt slot."""
    by_low = {h["chunk_id"].lower(): h for h in used}
    found = []
    for m in CITATION_RE.finditer(answer_text or ""):
        hit = by_low.get(m.group(1).lower())
        if hit is not None:
            found.append((m.end(), hit, "id"))
    for m in POSITIONAL_RE.finditer(answer_text or ""):
        n = int(m.group(1))
        if 1 <= n <= len(used):
            found.append((m.end(), used[n - 1], "position"))
    return sorted(found)


def jurisdiction_check(answer_text, used):
    """Catch the failure a real, in-prompt citation still hides: the answer claims one
    jurisdiction and cites another's case. Measured — given only ECHR passages, the model
    wrote "CH: ... (see [echr:001-242072:7])". Both id checks pass; the attribution is wrong,
    and in a comparative corpus that is the worst error available.

    Heuristic by construction: it reads the NEAREST jurisdiction label before each citation.
    """
    rows = []
    for end, hit, kind in resolve_citations(answer_text, used):
        labels = list(_JURIS_LABEL.finditer(answer_text, 0, end))
        if not labels:
            continue
        claimed = _JURIS_CANON.get(labels[-1].group(1).lower())
        if claimed and claimed != hit["jurisdiction"]:
            rows.append({"chunk_id": hit["chunk_id"], "claimed": claimed,
                         "actual": hit["jurisdiction"], "cited_as": kind,
                         "context": answer_text[labels[-1].start():end][-160:]})
    return rows


def grounding_report(result):
    """The three counts an answer must carry: retrieved, in-prompt, cited.

    They differ, and conflating them overstates the evidence base. `retrieved` is the recall
    set; `in_prompt` is what the generator could actually read (context budget); `cited` is
    what the answer text points at. Case counts are de-duplicated across chunks — 50 chunks
    are not 50 judgments.
    """
    hits, used = result.get("hits", []), result.get("used", [])
    forms = result.get("prompt_forms", {})
    cited = {c.lower() for c in CITATION_RE.findall(result.get("answer") or "")}
    by_low = {h["chunk_id"].lower(): h["chunk_id"] for h in used}
    resolved = resolve_citations(result.get("answer"), used)
    positional = {h["chunk_id"] for _, h, kind in resolved if kind == "position"}
    cited_chunks = sorted({orig for low, orig in by_low.items() if low in cited} | positional)
    # A citation the generator could not have read. Split by how bad it is: an id that exists
    # in the corpus but was not shown, vs an id that exists nowhere (invented outright).
    unmatched = [c for c in cited if c not in by_low]
    unseen = sorted(c for c in unmatched if c in _all_chunk_ids())
    invented = sorted(c for c in unmatched if c not in _all_chunk_ids())
    cited_cases = {_case_key(h) for h in used if h["chunk_id"] in set(cited_chunks)}
    return {
        "chunks_retrieved": len(hits),
        "cases_retrieved": len({_case_key(h) for h in hits}),
        "chunks_in_prompt": len(used),
        "cases_in_prompt": len({_case_key(h) for h in used}),
        "packing": GEN_PACKING,
        "chunks_in_prompt_full": sum(1 for h in used if forms.get(h["chunk_id"]) == "full"),
        "chunks_in_prompt_extract": sum(1 for h in used
                                        if forms.get(h["chunk_id"]) == "extract"),
        "chunks_cited": len(cited_chunks),
        "cases_cited": len(cited_cases),
        "cited_chunk_ids": cited_chunks,
        "citations_by_position": sorted(positional),   # cited as "[n]", resolved via prompt order
        "citations_not_in_prompt": sorted(unmatched),
        "citations_real_but_unseen": unseen,
        "citations_invented": invented,
        # the REAL packed prompt, not the sum of full chunk texts: under extract packing the
        # two differ by 4x, which made the budget look breached when it never was
        "prompt_chars": result.get("prompt_chars", sum(len(h["text"]) for h in used)),
        "jurisdiction_mismatches": jurisdiction_check(result.get("answer"), used),
        "jurisdictions_in_prompt": sorted({h["jurisdiction"] for h in used}),
        "jurisdictions_retrieved": sorted({h["jurisdiction"] for h in hits}),
        "score_range_in_prompt": ([round(used[-1]["score"], 4), round(used[0]["score"], 4)]
                                  if used else None),
    }


def _retrieved_entry(rank, h, query, used_ids, cited, forms):
    extracts, basis = kwic_extracts(h["text"], query)
    return {
        "rank": rank,
        "score": round(h["score"], 4),
        "in_prompt": h["chunk_id"] in used_ids,
        "prompt_form": forms.get(h["chunk_id"], "not_in_prompt"),
        "cited_in_answer": h["chunk_id"] in cited,
        "chunk_id": h["chunk_id"],
        "chunk_chars": len(h["text"]),
        "case": {"id": h["id"], "title": h["title"], "jurisdiction": h["jurisdiction"],
                 "source": h["source"], "country": h["country"], "date": h["date"],
                 "section": h["section"], "genre": h["genre"], "url": h["url"]},
        "extract_basis": basis,
        "extracts": extracts,
    }


def provenance_record(result, route="bucket1_retrieval_grounded"):
    """Full audit trail for one answer: parameters, what each filter removed, every retrieved
    chunk with its KWIC extract, and which of them the answer actually used and cited."""
    g = result.get("grounding") or grounding_report(result)
    cited = set(g["cited_chunk_ids"])
    used_ids = {h["chunk_id"] for h in result.get("used", [])}
    return {
        "asked_at": _dt_prov.now().isoformat(timespec="seconds"),
        "question": result["query"],
        "route": route,
        "mode": result["mode"],
        "abstained": result["abstained"],
        "abstain_type": result["abstain_type"],
        "route": result.get("route"),
        "scope": result.get("scope"),
        "answer": result["answer"],
        "params": {"emb_model": EMB_MODEL, "gen_model": GEN_MODEL,
                   "top_k": TOP_K, "score_margin": SCORE_MARGIN,
                   "score_floor_backstop": SCORE_FLOOR, "fetch_k": FETCH_K,
                   "max_chunks_per_case": MAX_CHUNKS_PER_CASE, "mmr_lambda": MMR_LAMBDA,
                   "genre_excluded": sorted(LAW_GENRE_EXCLUDE),
                   "gen_char_budget": GEN_CHAR_BUDGET, "gen_num_ctx": GEN_NUM_CTX,
                   "gen_answer_style": result.get("answer_style", GEN_ANSWER_STYLE),
                   "scope_by_respondent": SCOPE_BY_RESPONDENT,
                   "source_routing": SOURCE_ROUTING,
                   "gen_packing": GEN_PACKING, "gen_full_head": GEN_FULL_HEAD,
                   "gen_extract_window": GEN_EXTRACT_WINDOW,
                   "gen_extract_windows": GEN_EXTRACT_WINDOWS,
                   "kwic_window": KWIC_WINDOW, "corpus_chunks": len(chunks)},
        "comparative_sources": result.get("comparative_sources"),
        "recall": result.get("retrieval_stats", {}),
        "grounding": g,
        "retrieved": [_retrieved_entry(n, h, result["query"], used_ids, cited,
                                      result.get("prompt_forms", {}))
                      for n, h in enumerate(result.get("hits", []), 1)],
        "routed_out_communicated": [
            {"chunk_id": h["chunk_id"], "id": h["id"], "title": h["title"],
             "score": round(h["score"], 4)} for h in result.get("context", [])],
        "reading_notes": [
            "chunks_retrieved counts passages, cases_retrieved counts distinct judgments.",
            "rank is MMR selection order (relevance traded against diversity), not raw cosine "
            "order -- a lower-ranked chunk can carry a higher score.",
            "in_prompt=false means the chunk was retrieved but did not fit the generation "
            "context budget, so it cannot have grounded the answer.",
            "prompt_form=extract means the generator saw only the anchored windows of that "
            "chunk (sentence-aligned, '...' marks omitted text), not the whole passage -- so "
            "it grounds only what those windows carried. prompt_form=full means whole chunk.",
            "cited_in_answer=true means the answer text names that chunk id. "
            "citations_real_but_unseen = ids that exist in the corpus but were not in the "
            "prompt; citations_invented = ids that exist nowhere. Both are unfaithful.",
            "jurisdiction_mismatches lists citations where the answer claims one jurisdiction "
            "and cites another's case -- both id checks pass, the attribution is still wrong.",
            "extract_basis=no_lexical_anchor means the chunk matched semantically with no "
            "shared query word; the extract is then the head of the chunk, not a KWIC hit.",
            "scope.countries non-empty means the question named respondent State(s) and "
            "retrieval was RESTRICTED to them (recall.dropped_out_of_scope counts what that "
            "removed). The answer is then about those States only, by construction.",
        ],
    }


def save_provenance(result, route="bucket1_retrieval_grounded", outdir=PROVENANCE_DIR):
    rec = provenance_record(result, route=route)
    outdir.mkdir(parents=True, exist_ok=True)
    slug = _re_prov.sub(r"[^a-z0-9]+", "-", result["query"].lower())[:60].strip("-")
    path = outdir / f"{_dt_prov.now():%Y%m%dT%H%M%S}_{slug or 'question'}.json"
    path.write_text(_json_prov.dumps(rec, ensure_ascii=False, indent=1))
    return path


print(f"provenance ready — KWIC +/-{KWIC_WINDOW} chars + retrieved/in-prompt/cited audit "
      f"-> {PROVENANCE_DIR}/")

## 9. Display helper
Each line shows **date + jurisdiction + section**; then a sentence-aligned snippet.

In [ ]:
def _show_hit(n, h):
    date = h.get("date") or "n.d."
    print(f"  [{n}] cos={h['score']:.3f} | {h['jurisdiction']:9s} | {date:10s} | "
          f"{(h.get('country') or ''):4s} | {h.get('genre', ''):13s} | "
          f"{h.get('section', ''):10s} | {h['title'][:40]}")
    print(f"      {h['chunk_id']}  {h['url']}")
    print(f"      {h['snippet']}")


def show(result):
    print(f"Q: {result['query']}")
    line = f"mode={result['mode']}  abstained={result['abstained']}"
    if result["abstained"]:
        line += f"  type={result['abstain_type']}  ({result['abstain_reason']})"
    print(line)
    if result["answer"]:
        print(f"\nANSWER:\n{result['answer']}")
    print(f"\nRETRIEVED — substantive ({len(result['hits'])}):")
    for n, h in enumerate(result["hits"], 1):
        _show_hit(n, h)
    ctx = result.get("context") or []
    if ctx:
        print(f"\nCONTEXT — communicated cases routed out of the answer ({len(ctx)}):")
        for n, h in enumerate(ctx, 1):
            _show_hit(n, h)
    print("=" * 80)


print("show() ready")

show() ready


## 10. Demo — per-source queries (guarded)
Queries are phrased in each court's own register, which retrieves better than the abstract
label **"parental alienation"**. Caveat: in this corpus that label is polysemous — it also
matches *cultural/religious estrangement* cases (and the Austrian criminal sense of
*Entfremdung* = misappropriation), so concrete legal phrasings are used instead. The ECHR
hits should now come from the Court's assessment (`section=LAW`/`FACTS`), not quoted statutes.

In [ ]:
DEMO_QUERIES = [
    "Wann ist von einer Vollzugsmaßnahme abzusehen, wenn sie dem Kindeswohl widerspricht?",
    "Voraussetzungen für gemeinsame Obsorge bei fehlender Kommunikationsbasis der Eltern",
    "enforcement of contact rights where one parent obstructs the relationship with the other parent",
    "positive obligations of the State to maintain contact between parent and child",
    "Welche positiven Pflichten hat der Staat, um den Kontakt zwischen Elternteil und Kind aufrechtzuerhalten?",
    "Unter welchen Voraussetzungen kann einem Elternteil die Obhut über das Kind entzogen werden?",   # Swiss register (Obhut)
    "Wie wird das Besuchsrecht geregelt, wenn das Kind den Kontakt zum anderen Elternteil ablehnt?",
    "Wie viele Urteile betreffen die Durchsetzung des Kontaktrechts?",
]

if not chunks:
    print("Demo skipped — input data not found.")
    for _, p in SOURCES:
        print(f"   expected: {p}")
elif index is None:
    print("Demo skipped — index unavailable. pip install sentence-transformers faiss-cpu")
else:
    for q in DEMO_QUERIES:
        show(answer(q))

Q: Wann ist von einer Vollzugsmaßnahme abzusehen, wenn sie dem Kindeswohl widerspricht?
mode=generate  abstained=False

ANSWER:
Die Vollzugsmaßnahmen im Sinne des § 110 Abs 2 AußStrG iVm § 79 Abs 2 AußStrG sind gekennzeichnet durch die Verwirklichung des Leistungsbefehles unter Wahrung der Interessen aller Beteiligten, aber unter Hintansetzung von „schädigender Zweifelsucht und Ängstlichkeit".

Auch für das Absehen von einer Vollzugsmaßnahme ist ausschließlich das Kindeswohl maßgebliches Kriterium. Allerdings reicht ein bloßer Widerwille des Kindes gegen das Besuchsrecht ebensowenig wie ein Widerwille des anderen Elternteils aus, eine Gefährdung des Kindeswohles im Sinne des § 110 Abs 3 AußStrG zu bejahen.

Das Kindeswohl ist gefährdet, wenn die Vollzugsmaßnahme dem Kind selbst persönlichen Verkehr mit dessen Willen entzieht. Eine solche Gefährdung muss jedoch über die zwangsläufigen Folgen eines erneuten Aufenthaltswechsels hinausgehen.

Das Kindeswohl ist gefährdet, wenn eine Vollzug

## 11. Before / after — the ECHR boilerplate fix on the example query
The capability-boundary question is *"can the system tell when it has no law to state?"*. The
old top-k looked confident (cosine ~0.88) while being **all communicated boilerplate** —
high-similarity, zero-answer-value, near-duplicate. The fix routes that genre out of
law-questions and MMR-diversifies what remains, and — when *nothing but* boilerplate matches
— the new `type3_boilerplate_only` abstention says so honestly instead of paraphrasing a
pending application back as if it were settled law. Genre is also the field that makes the
*other* question answerable ("how often is this litigated") by including the same cases.

In [ ]:
EXAMPLE_QUERY = ("enforcement of contact rights where one parent obstructs the "
                 "relationship with the other parent")


def _print_ranking(label, hits):
    print(label)
    if not hits:
        print("    (none)")
    for n, h in enumerate(hits, 1):
        print(f"  [{n}] cos={h['score']:.3f} | {h.get('genre',''):13s} | "
              f"{h.get('section',''):10s} | {(h.get('country') or ''):4s} | {h['title'][:46]}")


if not chunks or index is None:
    print("Before/after skipped — data or index unavailable.")
    for _, p in SOURCES:
        if not p.exists():
            print(f"   missing: {p}")
else:
    # BEFORE: old behaviour — no genre filter, no MMR, plain top-6
    before = retrieve(EXAMPLE_QUERY, k=6, genre_filter=None, use_mmr=False)
    # AFTER: genre filter (drop communicated) + MMR diversity
    after = retrieve(EXAMPLE_QUERY, k=6)

    print(f"Q: {EXAMPLE_QUERY}\n")
    _print_ranking("BEFORE  (no genre filter, no MMR):", before)
    print()
    _print_ranking("AFTER   (exclude communicated + MMR):", after)

    b_comm = sum(1 for h in before if h.get("genre") == "communicated")
    b_docs = len({h["id"] for h in before})
    a_docs = len({h["id"] for h in after})
    print(f"\n  before: {b_comm}/6 communicated boilerplate, {b_docs} distinct documents")
    print(f"  after : {sum(1 for h in after if h.get('genre')=='communicated')}/6 communicated, "
          f"{a_docs} distinct documents")

    # genre breakdown of the ECHR corpus (documents)
    echr_recs = [r for r in records if r["source"] == "echr"]
    print("\nECHR documents by genre:",
          dict(Counter(assign_genre(r) for r in echr_recs)))

Q: enforcement of contact rights where one parent obstructs the relationship with the other parent

BEFORE  (no genre filter, no MMR):
  [1] cos=0.879 | communicated  | unparsed   | ROU  | CAERIDIN v. ROMANIA
  [2] cos=0.879 | communicated  | unparsed   | ROU  | TOIA v. ROMANIA
  [3] cos=0.877 | communicated  | unparsed   | ROU  | IONEL v. ROMANIA
  [4] cos=0.876 | communicated  | unparsed   | ROU  | BUȘ v. ROMANIA
  [5] cos=0.875 | communicated  | unparsed   | GRC  | ANAGNOSTAKIS v. GREECE
  [6] cos=0.874 | communicated  | unparsed   | NOR  | D.R. v. NORWAY

AFTER   (exclude communicated + MMR):
  [1] cos=0.870 | admissibility | HEADER     | DEU  | BUSSMANN v. GERMANY
  [2] cos=0.866 | merits        | FACTS      | POL  | CASE OF MALEC v. POLAND
  [3] cos=0.864 | merits        | LAW        | ROU  | CASE OF CRISTESCU v. ROMANIA
  [4] cos=0.863 | merits        | FACTS      | DEU  | CASE OF BUCHLEITHER v. GERMANY
  [5] cos=0.869 | merits        | LAW        | DEU  | CASE OF HOPPE v. GERMA